<a href="https://colab.research.google.com/github/StathisDevves/EV-smart-charging-V2G/blob/main/EV_P2P_V2G_Colab_Self_Contained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV–P2P–V2G UC1/UC2 — self-contained resumable Colab pipeline

This notebook contains the complete Python exporter. **No code ZIP is required.**
Only these two model inputs are uploaded:

1. `Input Data(8).xlsx`
2. `EV_P2P_V2G_Numbered_Equations_and_Excel_Specification (1).docx`

The pipeline produces six 8,760-hour workbooks (UC1/UC2 × modes 0/1/2), one comparative-summary workbook, progress reports, and persistent checkpoints in Google Drive. If execution is interrupted, rerun the notebook and use the same Drive account; `--resume` skips valid completed checkpoints and workbooks.


## 1. Upload or locate the two inputs

You may either run the cell below and choose the two files, or upload them beforehand through the Colab **Files** panel. If valid files already exist in `/content`, the cell will use the most recently uploaded matching files and will not ask again.


In [4]:
from pathlib import Path
from google.colab import files

content_dir = Path('/content')

def newest_match(pattern):
    candidates = [p for p in content_dir.glob(pattern) if p.is_file()]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

input_xlsx = newest_match('Input Data*.xlsx')
spec_docx = newest_match('EV_P2P_V2G_Numbered_Equations*.docx')

if input_xlsx is None or spec_docx is None:
    print('Upload exactly these two files:')
    print('  1) Input Data(8).xlsx')
    print('  2) EV_P2P_V2G_Numbered_Equations_and_Excel_Specification (1).docx')
    files.upload()
    input_xlsx = newest_match('Input Data*.xlsx')
    spec_docx = newest_match('EV_P2P_V2G_Numbered_Equations*.docx')

if input_xlsx is None or spec_docx is None:
    raise FileNotFoundError('Both the XLSX input and the M-EQ DOCX specification are required.')

print('XLSX input:', input_xlsx.name)
print('DOCX input:', spec_docx.name)


XLSX input: Input Data.xlsx
DOCX input: EV_P2P_V2G_Numbered_Equations_and_Excel_Specification (1).docx


## 2. Connect Google Drive

This cell asks for authorization only when Drive is not already mounted in the current runtime. Results and checkpoints are stored in `MyDrive/EV_P2P_V2G_Results`.


In [5]:
from google.colab import drive

drive_my = Path('/content/drive/MyDrive')
if drive_my.exists():
    print('Google Drive is already mounted.')
else:
    drive.mount('/content/drive')

results_dir = Path('/content/drive/MyDrive/EV_P2P_V2G_Results')
results_dir.mkdir(parents=True, exist_ok=True)
print('Persistent results/checkpoint directory:', results_dir)


Google Drive is already mounted.
Persistent results/checkpoint directory: /content/drive/MyDrive/EV_P2P_V2G_Results


## 3. Install dependencies and create the embedded-code directory

The following cells write the exporter modules already embedded in this notebook to the temporary Colab runtime. Nothing else needs to be uploaded.


In [6]:
%pip install -q "numpy>=1.24" "scipy>=1.10" "python-docx>=1.1" "openpyxl>=3.1" "XlsxWriter>=3.1"

import subprocess
import sys

code_dir = Path('/content/EV_P2P_V2G_Self_Contained_Code')
code_dir.mkdir(parents=True, exist_ok=True)
print('Embedded-code directory:', code_dir)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.4 MB/s eta 0:00:00
Embedded-code directory: /content/EV_P2P_V2G_Self_Contained_Code


## 4. Materialize the embedded exporter modules


In [7]:
%%writefile /content/EV_P2P_V2G_Self_Contained_Code/extract_docx_spec.py
from __future__ import annotations

import sys
from pathlib import Path

from docx import Document
from docx.document import Document as DocumentType
from docx.table import Table
from docx.text.paragraph import Paragraph


def iter_blocks(parent: DocumentType):
    """Yield paragraphs and tables in their original document order."""
    for child in parent.element.body.iterchildren():
        if child.tag.endswith("}p"):
            yield Paragraph(child, parent)
        elif child.tag.endswith("}tbl"):
            yield Table(child, parent)


def extract_text(source: Path) -> str:
    document = Document(source)
    lines: list[str] = []
    for block in iter_blocks(document):
        if isinstance(block, Paragraph):
            text = " ".join(block.text.split())
            if text:
                lines.append(text)
            continue

        for row in block.rows:
            cells: list[str] = []
            for cell in row.cells:
                paragraphs = [" ".join(p.text.split()) for p in cell.paragraphs]
                cells.append(" | ".join(p for p in paragraphs if p))
            if any(cells):
                lines.append(" | ".join(cells))
    return "\n".join(lines) + "\n"


def main() -> None:
    if len(sys.argv) != 3:
        raise SystemExit("Usage: extract_docx_spec.py <source.docx> <output.txt>")
    source = Path(sys.argv[1]).resolve()
    destination = Path(sys.argv[2]).resolve()
    if not source.is_file():
        raise SystemExit(f"DOCX not found: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(extract_text(source), encoding="utf-8")
    print(f"Extracted specification: {destination}")


if __name__ == "__main__":
    main()


Writing /content/EV_P2P_V2G_Self_Contained_Code/extract_docx_spec.py


In [8]:
%%writefile /content/EV_P2P_V2G_Self_Contained_Code/extract_ev_sources_colab.py
from __future__ import annotations

import json
import sys
from datetime import date, datetime, time
from pathlib import Path
from typing import Any

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.datetime import to_excel


REQUIRED_SHEETS = ("Monthly_Profile", "Solar_8760_2025")


def json_value(value: Any) -> Any:
    if isinstance(value, (datetime, date, time)):
        return float(to_excel(value))
    return value


def extract_workbook(source: Path) -> dict[str, dict[str, Any]]:
    values_wb = load_workbook(source, data_only=True, read_only=True)
    formula_wb = load_workbook(source, data_only=False, read_only=True)
    result: dict[str, dict[str, Any]] = {}

    for sheet_name in REQUIRED_SHEETS:
        if sheet_name not in values_wb.sheetnames:
            raise ValueError(f"Missing required sheet: {sheet_name}")
        values_ws = values_wb[sheet_name]
        formula_ws = formula_wb[sheet_name]
        values = [
            [json_value(v) for v in row]
            for row in values_ws.iter_rows(values_only=True)
        ]
        formulas = []
        for row in formula_ws.iter_rows():
            formulas.append([
                cell.value if cell.data_type == "f" else ""
                for cell in row
            ])
        result[sheet_name] = {
            "address": f"A1:{get_column_letter(values_ws.max_column)}{values_ws.max_row}",
            "values": values,
            "formulas": formulas,
        }

    hourly_rows = len(result["Solar_8760_2025"]["values"]) - 1
    if hourly_rows != 8760:
        raise ValueError(f"Expected 8,760 hourly rows, found {hourly_rows}")
    return result


def main() -> None:
    if len(sys.argv) != 3:
        raise SystemExit("Usage: extract_ev_sources_colab.py <input.xlsx> <output.json>")
    source = Path(sys.argv[1]).resolve()
    destination = Path(sys.argv[2]).resolve()
    if not source.is_file():
        raise SystemExit(f"Input workbook not found: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = extract_workbook(source)
    destination.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    print(f"Extracted 8,760-hour inputs: {destination}", flush=True)


if __name__ == "__main__":
    main()


Writing /content/EV_P2P_V2G_Self_Contained_Code/extract_ev_sources_colab.py


In [9]:
%%writefile /content/EV_P2P_V2G_Self_Contained_Code/prepare_ev_model.py
from __future__ import annotations

import argparse
import hashlib
import json
import math
import re
import sys
import time
from datetime import datetime, timezone
from itertools import combinations, permutations
from pathlib import Path

import numpy as np
from scipy.optimize import linprog


PLAYERS = ("P", "E", "D")


def decode_excel_serial(v: float) -> float:
    return float(v)


def coalition_key(players) -> str:
    s = "".join(sorted(players))
    return s or "empty"


def powerset(players=PLAYERS):
    for n in range(len(players) + 1):
        for c in combinations(players, n):
            yield frozenset(c)


def shapley(v):
    n = len(PLAYERS)
    fact = math.factorial
    ans = {p: 0.0 for p in PLAYERS}
    for p in PLAYERS:
        others = [x for x in PLAYERS if x != p]
        for s in powerset(others):
            w = fact(len(s)) * fact(n - len(s) - 1) / fact(n)
            ans[p] += w * (v[s | {p}] - v[s])
    return ans


def tau_value(v):
    N = frozenset(PLAYERS)
    utopia = {p: v[N] - v[N - {p}] for p in PLAYERS}
    minimum = {}
    for p in PLAYERS:
        vals = []
        for s in powerset():
            if p not in s:
                continue
            vals.append(v[s] - sum(utopia[j] for j in s if j != p))
        minimum[p] = max(vals)
    num = v[N] - sum(minimum.values())
    den = sum(utopia.values()) - sum(minimum.values())
    alpha = num / den if abs(den) > 1e-12 else 0.0
    alloc = {p: minimum[p] + alpha * (utopia[p] - minimum[p]) for p in PLAYERS}
    return {"utopia": utopia, "minimum_rights": minimum, "alpha": alpha, "allocation": alloc, "valid": 0 <= alpha <= 1}


def core_feasible(v):
    N = frozenset(PLAYERS)
    A_ub, b_ub = [], []
    for s in powerset():
        if not s or s == N:
            continue
        A_ub.append([-1.0 if p in s else 0.0 for p in PLAYERS])
        b_ub.append(-v[s])
    res = linprog(np.zeros(3), A_ub=A_ub, b_ub=b_ub, A_eq=[np.ones(3)], b_eq=[v[N]], bounds=[(None, None)] * 3, method="highs")
    return bool(res.success)


def average_lexicographic(v):
    if not core_feasible(v):
        return {"core_nonempty": False, "allocation": None}
    N = frozenset(PLAYERS)
    allocations = []
    for order in permutations(PLAYERS):
        fixed_A, fixed_b = [np.ones(3)], [v[N]]
        x_last = None
        ok = True
        for p in order:
            c = np.zeros(3)
            c[PLAYERS.index(p)] = -1
            A_ub, b_ub = [], []
            for s in powerset():
                if not s or s == N:
                    continue
                A_ub.append([-1.0 if j in s else 0.0 for j in PLAYERS])
                b_ub.append(-v[s])
            res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=fixed_A, b_eq=fixed_b, bounds=[(None, None)] * 3, method="highs")
            if not res.success:
                ok = False
                break
            x_last = res.x
            row = np.zeros(3)
            row[PLAYERS.index(p)] = 1
            fixed_A.append(row)
            fixed_b.append(res.x[PLAYERS.index(p)])
        if ok and x_last is not None:
            allocations.append(x_last)
    if not allocations:
        return {"core_nonempty": True, "allocation": None}
    avg = np.mean(np.vstack(allocations), axis=0)
    return {"core_nonempty": True, "allocation": {p: float(avg[i]) for i, p in enumerate(PLAYERS)}}


def convexity(v):
    failures = []
    for p in PLAYERS:
        remaining = [x for x in PLAYERS if x != p]
        subsets = list(powerset(remaining))
        for s in subsets:
            for t in subsets:
                if not s.issubset(t):
                    continue
                small = v[s | {p}] - v[s]
                large = v[t | {p}] - v[t]
                if small > large + 1e-8:
                    failures.append({"player": p, "S": coalition_key(s), "T": coalition_key(t), "small": small, "large": large})
    return {"is_convex": not failures, "failures": failures}


def parse_equations(spec_text: str):
    lines = [x.strip() for x in spec_text.splitlines() if x.strip()]
    eqs = []
    pat = re.compile(r"^(M-EQ\d{2})\s+([^|]+)\|\s*(.+?)\s*\|\s*Purpose:\s*(.+?)\s*\|\s*Implementation:\s*(.+?)\s*\|\s*Literature basis:\s*(.*?)(?:\tM-EQ\d{2})?$")
    for line in lines:
        m = pat.match(line)
        if m:
            eqs.append({"id": m.group(1), "title": m.group(2).strip(), "formula": m.group(3).strip(), "purpose": m.group(4).strip(), "implementation": m.group(5).strip(), "literature": m.group(6).strip()})
    extras = [
        {"id":"M-EQ63","title":"Convex network and production-externality cost","formula":"C_net,t = a Q_t + (b/2)Q_t^2 + (kappa/2)[Q_t-Qbar]_+^2","purpose":"Values increasing marginal network, loss and congestion costs.","implementation":"Excel formula based on hourly community grid import and the editable network coefficients.","literature":"Increasing marginal costs and cooperative production context: Moulin, Cooperative Microeconomics, Sections 5.1–5.2."},
        {"id":"M-EQ64","title":"Marginal network external cost","formula":"MC_net,t = a + b Q_t + kappa[Q_t-Qbar]_+","purpose":"Reports the marginal network cost of one additional kWh of grid import.","implementation":"Excel derivative of M-EQ63.","literature":"Marginal-cost pricing and production externalities: Moulin, Cooperative Microeconomics, Sections 5.1–5.2."},
        {"id":"M-EQ65","title":"Avoided production externality","formula":"AE_t = (C_net,t^BL + D_CO2,t^BL) - (C_net,t + D_CO2,t)","purpose":"Measures avoided network and carbon external costs relative to mode m=0.","implementation":"Excel baseline-minus-current formula using aligned hourly m=0 reference values.","literature":"Production-externality accounting and avoided-cost valuation; Moulin, Cooperative Microeconomics, Chapter 5."},
    ]
    by_id = {e["id"]: e for e in eqs}
    for e in extras:
        by_id[e["id"]] = e
    return [by_id[f"M-EQ{i:02d}"] for i in range(1, 66)]


ASSUMPTIONS = {
    "Year": 2025,
    "Delta_t_h": 1.0,
    "Rolling_Horizon_h": 24,
    "Retail_Adder_EUR_kWh": 0.12,
    "Minimum_Grid_Buy_EUR_kWh": 0.08,
    "Grid_Export_Multiplier": 0.80,
    "Minimum_Grid_Sell_EUR_kWh": 0.03,
    "Residential_PV_Scale": 1.0,
    "Commercial_PV_Scale": 2.0,
    "Residential_Load_Scale": 1.0,
    "Commercial_Load_Scale": 5.0,
    "Commercial_Day_Factor": 1.5,
    "Commercial_Night_Factor": 0.5,
    "Emission_Factor_kg_kWh": 0.35,
    "Carbon_Price_EUR_t": 80.0,
    "PV_Operating_Cost_EUR_kWh": 0.01,
    "Residential_BESS_Degradation_EUR_kWh": 0.04,
    "Commercial_BESS_Degradation_EUR_kWh": 0.04,
    "Network_Linear_Cost_EUR_kWh": 0.02,
    "Network_Quadratic_Cost_EUR_kW2h": 0.001,
    "Network_Congestion_Penalty_EUR_kW2h": 0.02,
    "Network_Dispatch_Overload_Cost_EUR_kWh": 1.0,
    "Network_Charge_EUR_kWh": 0.04,
    "Bargaining_Beta_Seller": 0.50,
    "EV_WTP_EUR_kWh": 0.30,
    "EV_Mobility_Value_EUR_km": 0.30,
    "Building_Energy_Utility_EUR_kWh": 0.30,
    "EV_Inconvenience_Zeta_EUR_kWh2": 0.005,
    "Participation_Cost_EUR_Session": 0.20,
    "Required_Participation_Premium_EUR_Session": 0.10,
    "V2G_Availability_Rate_EUR_kW_h": 0.01,
    "V2G_Activation_Rate_EUR_kWh": 0.10,
    "VOLL_EUR_kWh": 10.0,
    "Critical_Load_Share": 0.50,
    "Fuzzy_Membership_Mu": 1.0,
    "Price_Threshold_Gamma": 1.0,
    "Residential_BESS_Capacity_kWh": 15.0,
    "Commercial_BESS_Capacity_kWh": 30.0,
    "Residential_BESS_Power_kW": 5.0,
    "Commercial_BESS_Power_kW": 10.0,
    "Residential_Grid_Capacity_kW": 10.0,
    "Commercial_Grid_Capacity_UC1_kW": 25.0,
    "Commercial_Grid_Capacity_UC2_kW": 20.0,
    "Private_EV_Capacity_kWh": 60.0,
    "Private_EV_Power_kW": 7.4,
    "Private_EV_Consumption_kWh_km": 0.20,
    "Private_EV_Min_SOC": 0.20,
    "Private_EV_Max_SOC": 0.90,
    "Private_EV_Preferred_SOC": 0.80,
    "Private_EV_Buffer_kWh": 12.0,
    "Private_EV_Replacement_Cost_EUR": 9000.0,
    "Private_EV_Cycle_Life": 1500.0,
    "Taxi_EV_Capacity_kWh": 90.0,
    "Taxi_EV_Power_kW": 22.0,
    "Taxi_EV_Consumption_kWh_km": 0.20,
    "Taxi_EV_Min_SOC": 0.10,
    "Taxi_EV_Max_SOC": 0.95,
    "Taxi_EV_Preferred_SOC": 0.85,
    "Taxi_EV_Buffer_kWh": 10.0,
    "Taxi_EV_Replacement_Cost_EUR": 13500.0,
    "Taxi_EV_Cycle_Life": 1500.0,
    "EV_Charge_Efficiency": 0.95,
    "EV_Discharge_Efficiency": 0.95,
    "BESS_Charge_Efficiency": 0.95,
    "BESS_Discharge_Efficiency": 0.95,
}


def case_definition(case_id):
    if case_id == "UC1":
        ev = {"capacity":60.0,"power":7.4,"consumption":0.20,"min_soc":0.20,"max_soc":0.90,"preferred_soc":0.80,"buffer":12.0,"replacement":9000.0,"cycles":1500.0,"label":"Private EV"}
        nodes = [
            {"name":"Residential","strategy":"B","bess_capacity":15.0,"bess_power":5.0,"grid_capacity":10.0,"pv_scale":1.0,"load_scale":1.0},
            {"name":"Commercial","strategy":"C","bess_capacity":30.0,"bess_power":10.0,"grid_capacity":25.0,"pv_scale":2.0,"load_scale":5.0},
        ]
    else:
        ev = {"capacity":90.0,"power":22.0,"consumption":0.20,"min_soc":0.10,"max_soc":0.95,"preferred_soc":0.85,"buffer":10.0,"replacement":13500.0,"cycles":1500.0,"label":"EV Taxi"}
        nodes = [
            {"name":"Commercial","strategy":"C","bess_capacity":30.0,"bess_power":10.0,"grid_capacity":20.0,"pv_scale":2.0,"load_scale":5.0},
        ]
    return ev, nodes


def schedule(case_id, hour):
    if case_id == "UC1":
        if 8 <= hour < 9:
            return "Travel", -1, 15.0, 1
        if 19 <= hour < 20:
            return "Travel", -1, 15.0, 1
        if 9 <= hour < 19:
            return "Commercial", 1, 0.0, 0
        return "Residential", 0, 0.0, 0
    if 7 <= hour < 14:
        return "Travel", -1, 200.0 / 7.0, 1 if hour == 7 else 0
    if 17 <= hour < 24:
        return "Travel", -1, 200.0 / 7.0, 1 if hour == 17 else 0
    return "Commercial", 0, 0.0, 0


def build_inputs(src):
    rows = src["Solar_8760_2025"]["values"]
    header = rows[0]
    data = rows[1:]
    ts = np.array([float(r[0]) for r in data])
    month = np.array([int(r[3]) for r in data])
    day = np.array([int(r[4]) for r in data])
    hour_label = np.array([int(r[5]) for r in data])
    hour = hour_label - 1
    solar = np.array([float(r[6] or 0) for r in data])
    pv_base = np.array([float(r[7] or 0) for r in data])
    load_base = np.array([float(r[8] or 0) for r in data])
    wholesale_mwh = np.array([float(r[9]) for r in data])
    wholesale = wholesale_mwh / 1000.0
    grid_buy = np.maximum(ASSUMPTIONS["Minimum_Grid_Buy_EUR_kWh"], wholesale + ASSUMPTIONS["Retail_Adder_EUR_kWh"])
    grid_sell = np.maximum(ASSUMPTIONS["Minimum_Grid_Sell_EUR_kWh"], ASSUMPTIONS["Grid_Export_Multiplier"] * wholesale)
    res_pv = pv_base
    com_pv = pv_base * ASSUMPTIONS["Commercial_PV_Scale"]
    res_load = load_base
    factor = np.where((hour >= 8) & (hour < 20), ASSUMPTIONS["Commercial_Day_Factor"], ASSUMPTIONS["Commercial_Night_Factor"])
    com_load = load_base * ASSUMPTIONS["Commercial_Load_Scale"] * factor
    return {"source_header": header, "source_rows": data, "timestamp":ts,"month":month,"day":day,"hour_label":hour_label,"hour":hour,"solar":solar,"pv_base":pv_base,"load_base":load_base,"wholesale_mwh":wholesale_mwh,"wholesale":wholesale,"grid_buy":grid_buy,"grid_sell":grid_sell,"residential_pv":res_pv,"commercial_pv":com_pv,"residential_load":res_load,"commercial_load":com_load}


def simulate(case_id, mode, inp, der=True, progress_every=500):
    ev, nodes = case_definition(case_id)
    T = len(inp["timestamp"])
    N = len(nodes)
    pv = np.zeros((N, T))
    load = np.zeros((N, T))
    for n, node in enumerate(nodes):
        if node["name"] == "Residential":
            pv[n] = inp["residential_pv"] if der else 0
            load[n] = inp["residential_load"]
        else:
            pv[n] = inp["commercial_pv"] if der else 0
            load[n] = inp["commercial_load"]
    location_name, location_idx, distance, departure = [], [], np.zeros(T), np.zeros(T, dtype=int)
    for t, h in enumerate(inp["hour"]):
        name, idx, d, dep = schedule(case_id, int(h))
        location_name.append(name)
        location_idx.append(idx)
        distance[t] = d
        departure[t] = dep
    location_idx = np.array(location_idx)
    trip = distance * ev["consumption"]
    connected = (location_idx >= 0).astype(int)
    session_start = connected * (1 - np.roll(connected, 1))
    session_start[0] = 1 if connected[0] else 0

    H = int(ASSUMPTIONS["Rolling_Horizon_h"])
    eta_c = ASSUMPTIONS["EV_Charge_Efficiency"]
    eta_d = ASSUMPTIONS["EV_Discharge_Efficiency"]
    eta_bc = ASSUMPTIONS["BESS_Charge_Efficiency"]
    eta_bd = ASSUMPTIONS["BESS_Discharge_Efficiency"]
    ev_min, ev_max = ev["capacity"] * ev["min_soc"], ev["capacity"] * ev["max_soc"]
    ev_pref = ev["capacity"] * ev["preferred_soc"]
    ev_deg = ev["replacement"] / (2 * ev["capacity"] * ev["cycles"])
    bmins = np.array([0.10 * n["bess_capacity"] if der else 0.0 for n in nodes])
    bmaxs = np.array([0.90 * n["bess_capacity"] if der else 0.0 for n in nodes])
    bpowers = np.array([n["bess_power"] if der else 0.0 for n in nodes])
    bcaps = np.array([n["bess_capacity"] if der else 0.0 for n in nodes])
    gcaps = np.array([n["grid_capacity"] for n in nodes])
    bdeg = np.array([ASSUMPTIONS["Residential_BESS_Degradation_EUR_kWh"] if n["name"] == "Residential" else ASSUMPTIONS["Commercial_BESS_Degradation_EUR_kWh"] for n in nodes])

    ev_state = ev_pref
    bess_state = 0.5 * bcaps
    out = {k: np.zeros(T) for k in ["ev_start","ev_ch","ev_dis","ev_end","v2g_reserve","bess_start","bess_ch","bess_dis","bess_end","grid_import","grid_export","overload"]}
    node_out = {k: np.zeros((N,T)) for k in ["bess_start","bess_ch","bess_dis","bess_end","grid_import","grid_export","overload"]}

    def solve_hour(t, ev_state0, bess_state0):
        idxs = np.array([(t + k) % T for k in range(H)], dtype=int)
        # variable blocks, each H or N*H
        cursor = 0
        def block(size):
            nonlocal cursor
            a = np.arange(cursor, cursor + size).reshape((-1,))
            cursor += size
            return a
        bch = block(N*H).reshape(N,H)
        bdis = block(N*H).reshape(N,H)
        gimp = block(N*H).reshape(N,H)
        gexp = block(N*H).reshape(N,H)
        ov = block(N*H).reshape(N,H)
        ech = block(H)
        edis = block(H)
        eend = block(H)
        bend = block(N*H).reshape(N,H)
        nv = cursor
        c = np.zeros(nv)
        bounds = [(0, None)] * nv

        # Greedy uncontrolled forecast for m=0.
        fixed_ch = np.zeros(H)
        pred = ev_state0
        if mode == 0:
            for k, ix in enumerate(idxs):
                if connected[ix]:
                    fixed_ch[k] = min(ev["power"], max(0.0, (ev_pref - pred) / eta_c))
                pred = pred + eta_c * fixed_ch[k] - trip[ix]

        for k, ix in enumerate(idxs):
            carbon = ASSUMPTIONS["Carbon_Price_EUR_t"] * ASSUMPTIONS["Emission_Factor_kg_kWh"] / 1000.0
            for n in range(N):
                c[gimp[n,k]] = inp["grid_buy"][ix] + carbon + ASSUMPTIONS["Network_Linear_Cost_EUR_kWh"]
                c[gexp[n,k]] = -inp["grid_sell"][ix]
                c[ov[n,k]] = ASSUMPTIONS["Network_Dispatch_Overload_Cost_EUR_kWh"]
                c[bch[n,k]] = bdeg[n]
                c[bdis[n,k]] = bdeg[n]
                bounds[bch[n,k]] = (0, bpowers[n])
                bounds[bdis[n,k]] = (0, bpowers[n])
                bounds[gimp[n,k]] = (0, None)
                bounds[gexp[n,k]] = (0, None)
                bounds[ov[n,k]] = (0, None)
                bounds[bend[n,k]] = (bmins[n], bmaxs[n])
            c[ech[k]] = ev_deg
            c[edis[k]] = ev_deg
            if mode == 0:
                bounds[ech[k]] = (fixed_ch[k], fixed_ch[k])
                bounds[edis[k]] = (0, 0)
            elif mode == 1:
                bounds[ech[k]] = (0, ev["power"] if connected[ix] else 0)
                bounds[edis[k]] = (0, 0)
            else:
                bounds[ech[k]] = (0, ev["power"] if connected[ix] else 0)
                bounds[edis[k]] = (0, ev["power"] if connected[ix] else 0)
            bounds[eend[k]] = (ev_min, ev_max)

        Aeq, beq = [], []
        for k, ix in enumerate(idxs):
            # node balances
            for n in range(N):
                row = np.zeros(nv)
                row[gimp[n,k]] = 1
                row[gexp[n,k]] = -1
                row[bdis[n,k]] = 1
                row[bch[n,k]] = -1
                if location_idx[ix] == n:
                    row[edis[k]] = 1
                    row[ech[k]] = -1
                Aeq.append(row)
                beq.append(float(load[n,ix] - pv[n,ix]))
            # BESS transitions
            for n in range(N):
                row = np.zeros(nv)
                row[bend[n,k]] = 1
                row[bch[n,k]] = -eta_bc
                row[bdis[n,k]] = 1/eta_bd
                if k > 0:
                    row[bend[n,k-1]] = -1
                    rhs = 0.0
                else:
                    rhs = float(bess_state0[n])
                Aeq.append(row)
                beq.append(rhs)
            # EV transition
            row = np.zeros(nv)
            row[eend[k]] = 1
            row[ech[k]] = -eta_c
            row[edis[k]] = 1/eta_d
            if k > 0:
                row[eend[k-1]] = -1
                rhs = -float(trip[ix])
            else:
                rhs = float(ev_state0 - trip[ix])
            Aeq.append(row)
            beq.append(rhs)

        Aub, bub = [], []
        # Overload definition.
        for k in range(H):
            for n in range(N):
                row = np.zeros(nv)
                row[gimp[n,k]] = 1
                row[ov[n,k]] = -1
                Aub.append(row)
                bub.append(float(gcaps[n]))
        # Departure preference before each trip start in forecast.
        for k, ix in enumerate(idxs):
            if departure[ix] != 1:
                continue
            if k == 0:
                continue
            row = np.zeros(nv)
            row[eend[k-1]] = -1
            Aub.append(row)
            bub.append(-ev_pref)
        # No artificial EV terminal-SOC equality: departure constraints within
        # the rolling window protect mobility, while overnight charging may
        # legitimately begin immediately after the 24:00 taxi-shift boundary.
        for n in range(N):
            row = np.zeros(nv)
            row[bend[n,H-1]] = -1
            Aub.append(row)
            bub.append(-float(bess_state0[n]))

        res = linprog(c, A_ub=np.array(Aub), b_ub=np.array(bub), A_eq=np.array(Aeq), b_eq=np.array(beq), bounds=bounds, method="highs")
        if not res.success:
            raise RuntimeError(f"{case_id} mode {mode} t={t}: {res.message}")
        x = res.x
        return {
            "ev_ch":x[ech[0]],"ev_dis":x[edis[0]],"ev_end":x[eend[0]],
            "bch":x[bch[:,0]],"bdis":x[bdis[:,0]],"bend":x[bend[:,0]],
            "gimp":x[gimp[:,0]],"gexp":x[gexp[:,0]],"ov":x[ov[:,0]],
        }

    started = time.monotonic()
    run_name = f"{case_id} m={mode} {'main/DER' if der else 'counterfactual/no-DER'}"
    print(f"  {run_name}: starting {T:,} rolling-hour optimisations", flush=True)
    for t in range(T):
        sol = solve_hour(t, ev_state, bess_state)
        out["ev_start"][t] = ev_state
        out["ev_ch"][t] = sol["ev_ch"]
        out["ev_dis"][t] = sol["ev_dis"]
        out["ev_end"][t] = sol["ev_end"]
        out["bess_start"][t] = bess_state.sum()
        out["bess_ch"][t] = sol["bch"].sum()
        out["bess_dis"][t] = sol["bdis"].sum()
        out["bess_end"][t] = sol["bend"].sum()
        out["grid_import"][t] = sol["gimp"].sum()
        out["grid_export"][t] = sol["gexp"].sum()
        out["overload"][t] = sol["ov"].sum()
        for n in range(N):
            node_out["bess_start"][n,t] = bess_state[n]
            node_out["bess_ch"][n,t] = sol["bch"][n]
            node_out["bess_dis"][n,t] = sol["bdis"][n]
            node_out["bess_end"][n,t] = sol["bend"][n]
            node_out["grid_import"][n,t] = sol["gimp"][n]
            node_out["grid_export"][n,t] = sol["gexp"][n]
            node_out["overload"][n,t] = sol["ov"][n]
        ev_state = sol["ev_end"]
        bess_state = sol["bend"]
        completed = t + 1
        if completed == 1 or completed % progress_every == 0 or completed == T:
            elapsed = max(time.monotonic() - started, 1e-9)
            rate = completed / elapsed
            eta_seconds = (T - completed) / rate if rate > 0 else 0
            print(
                f"  {run_name}: {completed:,}/{T:,} hours "
                f"({100 * completed / T:5.1f}%), elapsed {elapsed / 60:6.1f} min, "
                f"ETA {eta_seconds / 60:6.1f} min",
                flush=True,
            )

    # Ex-post trade classification and accounting arrays.
    p2p_buy = np.zeros(T); p2p_sell = np.zeros(T); pv_self = np.zeros(T); surplus = np.zeros(T); deficit = np.zeros(T)
    for t in range(T):
        for n in range(N):
            evc = out["ev_ch"][t] if location_idx[t] == n else 0.0
            supply_before_ev = pv[n,t] + node_out["bess_dis"][n,t]
            demand_before_ev = load[n,t] + node_out["bess_ch"][n,t]
            pv_self[t] += min(pv[n,t], demand_before_ev + evc)
            s = max(0.0, supply_before_ev - demand_before_ev)
            d = max(0.0, demand_before_ev - supply_before_ev)
            surplus[t] += s; deficit[t] += d
            if location_idx[t] == n and mode >= 1:
                p2p_buy[t] = min(out["ev_ch"][t], s)
                if mode == 2:
                    p2p_sell[t] = min(out["ev_dis"][t], d)
    v2g_act = np.maximum(0.0, out["ev_dis"] - p2p_sell)
    reserve_floor = ev_min + ev["buffer"]
    v2g_reserve = connected * (mode == 2) * np.minimum(ev["power"], np.maximum(0.0, (out["ev_start"] - reserve_floor) * eta_d))
    out.update({"connected":connected.astype(float),"location":location_name,"location_idx":location_idx.astype(int),"distance":distance,"trip_energy":trip,"departure":departure.astype(float),"session_start":session_start.astype(float),"p2p_buy":p2p_buy,"p2p_sell":p2p_sell,"v2g_activation":v2g_act,"v2g_reserve":v2g_reserve,"pv_self":pv_self,"surplus":surplus,"deficit":deficit,"pv_total":pv.sum(axis=0),"load_total":load.sum(axis=0)})

    # Cost summary for coalition construction (same definitions as workbook formulas).
    grid_energy = float(np.sum(inp["grid_buy"]*out["grid_import"] - inp["grid_sell"]*out["grid_export"]))
    ev_degradation = float(np.sum((out["ev_ch"]+out["ev_dis"])*ev_deg))
    bess_degradation = float(sum(np.sum((node_out["bess_ch"][n]+node_out["bess_dis"][n])*bdeg[n]) for n in range(N)))
    pv_operating = float(np.sum(out["pv_total"])*ASSUMPTIONS["PV_Operating_Cost_EUR_kWh"])
    inconvenience = float(np.sum(0.5*ASSUMPTIONS["EV_Inconvenience_Zeta_EUR_kWh2"]*np.maximum(0, ev_pref-out["ev_end"])**2*connected))
    Q = out["grid_import"]
    qbar = sum(gcaps)
    network_hour = ASSUMPTIONS["Network_Linear_Cost_EUR_kWh"]*Q + 0.5*ASSUMPTIONS["Network_Quadratic_Cost_EUR_kW2h"]*Q**2 + 0.5*ASSUMPTIONS["Network_Congestion_Penalty_EUR_kW2h"]*np.maximum(0,Q-qbar)**2
    network = float(np.sum(network_hour))
    carbon_hour = ASSUMPTIONS["Carbon_Price_EUR_t"]*ASSUMPTIONS["Emission_Factor_kg_kWh"]*Q/1000
    carbon = float(np.sum(carbon_hour))
    total = grid_energy+ev_degradation+bess_degradation+pv_operating+inconvenience+network+carbon
    out["cost_summary"] = {"total":total,"noncarbon":total-carbon,"grid_energy":grid_energy,"ev_degradation":ev_degradation,"bess_degradation":bess_degradation,"pv_operating":pv_operating,"inconvenience":inconvenience,"network":network,"carbon":carbon}
    out["node_results"] = node_out
    return out


def make_coalition(case_modes, no_der_modes):
    base = no_der_modes[0]["cost_summary"]
    def val(sim):
        c = sim["cost_summary"]
        carbon_benefit = base["carbon"] - c["carbon"]
        return (base["noncarbon"] - c["noncarbon"]) + carbon_benefit
    mapping = {
        frozenset(): 0.0,
        frozenset("P"): val(case_modes[0]),
        frozenset("E"): val(no_der_modes[1]),
        frozenset("D"): 0.0,
        frozenset(("P","E")): val(case_modes[1]),
        frozenset(("P","D")): val(case_modes[0]),
        frozenset(("E","D")): val(no_der_modes[2]),
        frozenset(("P","E","D")): val(case_modes[2]),
    }
    costs = {coalition_key(s): base["total"] - mapping[s] for s in mapping}
    def package(v):
        return {"values":{coalition_key(s):float(v[s]) for s in v},"shapley":shapley(v),"tau":tau_value(v),"al":average_lexicographic(v),"convexity":convexity(v)}
    general = package(mapping)
    clan_v = {s:(mapping[s] if (not s or "D" in s) else 0.0) for s in mapping}
    general["clan"] = package(clan_v)
    general["clan"]["validity_nonnegative"] = all(clan_v[s] >= -1e-8 for s in clan_v if "D" in s)
    general["coalition_costs"] = costs
    general["baseline_cost"] = base["total"]
    return general


def to_jsonable(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, dict): return {str(k):to_jsonable(v) for k,v in obj.items()}
    if isinstance(obj, list): return [to_jsonable(v) for v in obj]
    return obj


def atomic_write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(to_jsonable(payload), separators=(",", ":")), encoding="utf-8")
    temporary.replace(path)


def model_fingerprint(input_path: Path, spec_path: Path) -> str:
    digest = hashlib.sha256()
    digest.update(b"EV-P2P-V2G-CHECKPOINT-V2\0")
    digest.update(input_path.read_bytes())
    digest.update(b"\0SPEC\0")
    digest.update(spec_path.read_bytes())
    digest.update(b"\0ASSUMPTIONS\0")
    digest.update(json.dumps(ASSUMPTIONS, sort_keys=True, separators=(",", ":")).encode("utf-8"))
    return digest.hexdigest()


def checkpoint_name(case_id: str, mode: int, der: bool) -> str:
    suffix = "der" if der else "no_der"
    return f"{case_id}_m{mode}_{suffix}.json"


def validate_checkpoint(payload, fingerprint: str, case_id: str, mode: int, der: bool, expected_hours: int):
    if not isinstance(payload, dict):
        return False, "payload is not an object"
    expected = {"checkpoint_version": 2, "fingerprint": fingerprint, "case_id": case_id, "mode": mode, "der": der}
    for key, value in expected.items():
        if payload.get(key) != value:
            return False, f"metadata mismatch: {key}"
    result = payload.get("result")
    if not isinstance(result, dict) or "cost_summary" not in result:
        return False, "missing result/cost_summary"
    if der:
        required = ("ev_start", "ev_ch", "ev_dis", "ev_end", "grid_import", "grid_export", "pv_total", "load_total")
        for key in required:
            if not isinstance(result.get(key), list) or len(result[key]) != expected_hours:
                return False, f"invalid hourly array: {key}"
    return True, "valid"


def load_or_run_checkpoint(
    checkpoint_dir: Path,
    fingerprint: str,
    case_id: str,
    mode: int,
    der: bool,
    inp,
    resume: bool,
    progress_every: int,
    position: int,
    total_runs: int,
):
    path = checkpoint_dir / checkpoint_name(case_id, mode, der)
    label = f"{case_id} m={mode} {'main/DER' if der else 'counterfactual/no-DER'}"
    if resume and path.exists():
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            valid, reason = validate_checkpoint(payload, fingerprint, case_id, mode, der, len(inp["timestamp"]))
            if valid:
                print(f"[{position}/{total_runs}] RESUME: {label} loaded from {path.name}", flush=True)
                return payload["result"], True
            print(f"[{position}/{total_runs}] IGNORE invalid checkpoint {path.name}: {reason}", flush=True)
        except Exception as exc:
            print(f"[{position}/{total_runs}] IGNORE unreadable checkpoint {path.name}: {exc}", flush=True)

    print(f"[{position}/{total_runs}] START: {label}", flush=True)
    result = simulate(case_id, mode, inp, der=der, progress_every=progress_every)
    stored_result = result if der else {"cost_summary": result["cost_summary"]}
    payload = {
        "checkpoint_version": 2,
        "fingerprint": fingerprint,
        "case_id": case_id,
        "mode": mode,
        "der": der,
        "completed_hours": len(inp["timestamp"]),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "result": stored_result,
    }
    atomic_write_json(path, payload)
    print(f"[{position}/{total_runs}] SAVED: {label} -> {path.name}", flush=True)
    return stored_result, False


def write_manifest(checkpoint_dir: Path, fingerprint: str, completed: list[str], total_runs: int) -> None:
    atomic_write_json(checkpoint_dir / "checkpoint_manifest.json", {
        "checkpoint_version": 2,
        "fingerprint": fingerprint,
        "completed": completed,
        "completed_count": len(completed),
        "total_runs": total_runs,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
    })


def main():
    parser = argparse.ArgumentParser(description="Build the EV–P2P–V2G model with resumable scenario checkpoints.")
    parser.add_argument("input_values", type=Path)
    parser.add_argument("spec_text", type=Path)
    parser.add_argument("output", type=Path)
    parser.add_argument("--checkpoint-dir", type=Path, default=None)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--progress-every", type=int, default=500)
    args = parser.parse_args()
    if args.progress_every < 1:
        parser.error("--progress-every must be at least 1")

    input_path = args.input_values.resolve()
    spec_path = args.spec_text.resolve()
    output_path = args.output.resolve()
    checkpoint_dir = (args.checkpoint_dir or output_path.parent / "checkpoints").resolve()
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    src = json.loads(input_path.read_text())
    inp = build_inputs(src)
    equations = parse_equations(spec_path.read_text(encoding="utf-8"))
    fingerprint = model_fingerprint(input_path, spec_path)
    print(f"Checkpoint directory: {checkpoint_dir}", flush=True)
    print(f"Model fingerprint: {fingerprint[:16]}…", flush=True)
    print(f"Resume enabled: {args.resume}", flush=True)
    result = {"assumptions":ASSUMPTIONS,"equations":equations,"source_monthly":src["Monthly_Profile"]["values"],"source_header":inp["source_header"],"source_rows":inp["source_rows"],"prepared_inputs":inp,"cases":{}}
    total_runs = 12
    position = 0
    completed = []
    for case_id in ("UC1","UC2"):
        modes = {}
        no_der = {}
        for m in (0,1,2):
            position += 1
            modes[m], _ = load_or_run_checkpoint(
                checkpoint_dir, fingerprint, case_id, m, True, inp,
                args.resume, args.progress_every, position, total_runs,
            )
            completed.append(checkpoint_name(case_id, m, True))
            write_manifest(checkpoint_dir, fingerprint, completed, total_runs)
        for m in (0,1,2):
            position += 1
            no_der[m], _ = load_or_run_checkpoint(
                checkpoint_dir, fingerprint, case_id, m, False, inp,
                args.resume, args.progress_every, position, total_runs,
            )
            completed.append(checkpoint_name(case_id, m, False))
            write_manifest(checkpoint_dir, fingerprint, completed, total_runs)
        ev,nodes = case_definition(case_id)
        result["cases"][case_id] = {"ev":ev,"nodes":nodes,"modes":modes,"no_der_modes":{m:{"cost_summary":no_der[m]["cost_summary"]} for m in no_der},"coalition":make_coalition(modes,no_der)}
        atomic_write_json(checkpoint_dir / f"partial_model_{case_id}.json", {**result, "checkpoint_fingerprint": fingerprint})
        print(f"ASSEMBLED: {case_id} partial model saved", flush=True)
    result["checkpoint_fingerprint"] = fingerprint
    atomic_write_json(output_path, result)
    print(f"COMPLETE: model written to {output_path}", flush=True)


if __name__ == "__main__":
    main()


Writing /content/EV_P2P_V2G_Self_Contained_Code/prepare_ev_model.py


In [10]:
%%writefile /content/EV_P2P_V2G_Self_Contained_Code/build_ev_workbooks_colab.py
from __future__ import annotations

import json
import math
import sys
from pathlib import Path
from statistics import fmean
from typing import Any, Iterable

import xlsxwriter
from xlsxwriter.utility import xl_col_to_name


HEADERS = [
    "Scenario_ID", "DateTime", "Month_ID [M-EQ62]", "Hour_Start",
    "Wholesale_EUR_kWh [M-EQ01]", "Grid_Buy_Price_EUR_kWh [INPUT]", "Grid_Sell_Price_EUR_kWh [INPUT]",
    "Community_PV_Generation_kWh [M-EQ02]", "Community_Building_Load_kWh [M-EQ03]",
    "EV_Connected [M-EQ04]", "EV_Location", "Distance_km [M-EQ05]", "Trip_Energy_kWh [M-EQ06]",
    "Departure_Flag [M-EQ07]", "Required_Departure_Energy_kWh [M-EQ08]", "EV_Energy_Start_kWh [OPTIMIZER]",
    "EV_Charge_Power_kW [DV01]", "EV_Discharge_Power_kW [DV02]", "EV_Energy_End_kWh [M-EQ09]",
    "EV_Bounds_Flag [M-EQ10]", "Community_BESS_Energy_Start_kWh [OPTIMIZER]", "Community_BESS_Charge_Power_kW [DV04]",
    "Community_BESS_Discharge_Power_kW [DV04]", "Community_BESS_Energy_End_kWh [M-EQ11]", "BESS_Bounds_Flag [M-EQ12]",
    "PV_Self_Consumption_kWh [M-EQ13]", "Tradable_Surplus_kWh [M-EQ14]", "Residual_Deficit_kWh [M-EQ14]",
    "Rolling_Wholesale_Mean_EUR_kWh [M-EQ15]", "Rolling_Wholesale_SD_EUR_kWh [M-EQ15]",
    "Low_Price_Threshold_EUR_kWh [M-EQ16]", "High_Price_Threshold_EUR_kWh [M-EQ16]",
    "Desired_EV_Charge_kWh [M-EQ17]", "P2P_EV_Buy_kWh [M-EQ18]", "Grid_EV_Buy_kWh [M-EQ19]",
    "P2P_EV_Sell_kWh [M-EQ20]", "V2G_Reserve_kW [M-EQ21]", "V2G_Activated_kWh [M-EQ22]",
    "Mutual_Exclusion_Flag [M-EQ23]", "EV_Mode [M-EQ24]", "Net_Node_Exchange_kWh [M-EQ25]",
    "Community_Grid_Import_kWh [M-EQ26]", "Community_Grid_Export_kWh [M-EQ26]", "Balance_Residual_kWh [M-EQ26]",
    "Seller_Reservation_EUR_kWh [M-EQ27]", "Buyer_Reservation_EUR_kWh [M-EQ28]", "P2P_Price_EUR_kWh [M-EQ29]",
    "Trade_Feasible_Flag [M-EQ30]", "P2P_Trade_Surplus_EUR [M-EQ31]", "V2G_Availability_Revenue_EUR [M-EQ32]",
    "V2G_Activation_Revenue_EUR [M-EQ33]", "V2G_Remuneration_Margin_EUR [M-EQ34]", "EV_Degradation_Cost_EUR [M-EQ35]",
    "EV_Inconvenience_Cost_EUR [M-EQ36]", "Participation_Cost_EUR [M-EQ37]", "EV_Net_Benefit_EUR [M-EQ38]",
    "Prosumer_Net_Benefit_EUR [M-EQ39]", "DSO_Net_Benefit_EUR [M-EQ40]", "Grid_Related_CO2_kg [M-EQ41]",
    "Carbon_Externality_Cost_EUR [M-EQ42]", "Community_Real_System_Cost_EUR [M-EQ43]", "Community_Social_Welfare_EUR [M-EQ44]",
    "Transfer_Check_EUR [M-EQ45]", "Unserved_Load_kWh [M-EQ53]", "Outage_Cost_EUR [M-EQ54]",
    "EFC_Contribution [M-EQ56]", "IR_Flag [M-EQ60]", "Convex_Network_Cost_EUR [M-EQ63]",
    "Marginal_Network_Cost_EUR_kWh [M-EQ64]", "Avoided_Production_Externality_EUR [M-EQ65]",
]

DECISION_COLS = {15, 16, 17, 20, 21, 22, 33, 35, 36, 37, 39, 41, 42}
FORMULA_COLS = set(range(len(HEADERS))) - DECISION_COLS - {0, 1, 3}

COLORS = {
    "navy": "#17324D", "teal": "#0F766E", "blue": "#DCEAF7",
    "pale": "#F5F7FA", "input": "#FFF2CC", "decision": "#FCE4D6",
    "formula": "#E2F0D9", "white": "#FFFFFF", "line": "#CBD5E1",
    "good": "#D9EAD3", "bad": "#F4CCCC",
}


def mode_description(mode: int) -> str:
    return [
        "Grid-only / non-participation",
        "Smart charging + P2P buying",
        "Smart charging + P2P buying/selling + V2G",
    ][mode]


def total(values: Iterable[float]) -> float:
    return float(sum(float(x or 0) for x in values))


def enrich_assumptions(model: dict[str, Any], case_id: str, mode: int) -> dict[str, Any]:
    case = model["cases"][case_id]
    ev, nodes = case["ev"], case["nodes"]
    a = dict(model["assumptions"])
    a.update({
        "Case_ID": case_id,
        "EV_Mode": mode,
        "EV_Mode_Description": mode_description(mode),
        "EV_Label": ev["label"],
        "Community_PV_Scale": total(n["pv_scale"] for n in nodes),
        "Community_Grid_Capacity_kW": total(n["grid_capacity"] for n in nodes),
        "Community_BESS_Capacity_kWh": total(n["bess_capacity"] for n in nodes),
        "Community_BESS_Power_kW": total(n["bess_power"] for n in nodes),
        "Community_BESS_Min_Energy_kWh": total(0.10 * n["bess_capacity"] for n in nodes),
        "Community_BESS_Max_Energy_kWh": total(0.90 * n["bess_capacity"] for n in nodes),
        "EV_Capacity_kWh": ev["capacity"],
        "EV_Power_kW": ev["power"],
        "EV_Consumption_kWh_km": ev["consumption"],
        "EV_Min_Energy_kWh": ev["capacity"] * ev["min_soc"],
        "EV_Max_Energy_kWh": ev["capacity"] * ev["max_soc"],
        "EV_Preferred_Energy_kWh": ev["capacity"] * ev["preferred_soc"],
        "EV_Buffer_kWh": ev["buffer"],
        "EV_Replacement_Cost_EUR": ev["replacement"],
        "EV_Cycle_Life": ev["cycles"],
        "EV_Degradation_EUR_kWh": ev["replacement"] / (2 * ev["capacity"] * ev["cycles"]),
        "Next_Trip_Energy_kWh": 3.0 if case_id == "UC1" else 40.0,
    })
    outside = outside_option(model, case_id, a)
    a.update({
        "Outside_Option_m0_EV_Benefit_EUR": outside["m0"],
        "Outside_Option_Smart_Grid_EV_Benefit_EUR": outside["smart_grid_only"],
        "Strongest_Outside_Option_EV_Benefit_EUR": outside["strongest"],
        "Prosumer_Strategies": "; ".join(
            f"{n['name']}: Strategy {n['strategy']}, {n['bess_capacity']} kWh BESS" for n in nodes
        ),
    })
    return a


def outside_option(model: dict[str, Any], case_id: str, a: dict[str, Any]) -> dict[str, float]:
    case, prepared = model["cases"][case_id], model["prepared_inputs"]
    candidates = []
    for mode in (0, 1):
        sim = case["modes"][str(mode)]
        benefit = 0.0
        for i in range(8760):
            inconvenience = (
                0.5 * a["EV_Inconvenience_Zeta_EUR_kWh2"]
                * max(0.0, a["EV_Preferred_Energy_kWh"] - sim["ev_end"][i]) ** 2
                * sim["connected"][i]
            )
            benefit += (
                a["EV_Mobility_Value_EUR_km"] * sim["distance"][i]
                - prepared["grid_buy"][i] * sim["ev_ch"][i]
                - a["EV_Degradation_EUR_kWh"] * (sim["ev_ch"][i] + sim["ev_dis"][i])
                - inconvenience
            )
        candidates.append(benefit)
    return {"m0": candidates[0], "smart_grid_only": candidates[1], "strongest": max(candidates)}


def baseline_externality(model: dict[str, Any], case_id: str, a: dict[str, Any]) -> tuple[list[float], list[float]]:
    base = model["cases"][case_id]["modes"]["0"]
    network, carbon = [], []
    for q in base["grid_import"]:
        network.append(
            a["Network_Linear_Cost_EUR_kWh"] * q
            + 0.5 * a["Network_Quadratic_Cost_EUR_kW2h"] * q * q
            + 0.5 * a["Network_Congestion_Penalty_EUR_kW2h"]
            * max(0.0, q - a["Community_Grid_Capacity_kW"]) ** 2
        )
        carbon.append(a["Carbon_Price_EUR_t"] * a["Emission_Factor_kg_kWh"] * q / 1000)
    return network, carbon


def hourly_matrix(model: dict[str, Any], case_id: str, mode: int, a: dict[str, Any]) -> list[list[Any]]:
    case, prepared = model["cases"][case_id], model["prepared_inputs"]
    sim = case["modes"][str(mode)]
    base_network, base_carbon = baseline_externality(model, case_id, a)
    bess_deg = max(a["Residential_BESS_Degradation_EUR_kWh"], a["Commercial_BESS_Degradation_EUR_kWh"])
    seller_cost = a["PV_Operating_Cost_EUR_kWh"] + bess_deg
    rows, ev_benefits = [], []

    for i in range(8760):
        wh, buy, sell = prepared["wholesale"][i], prepared["grid_buy"][i], prepared["grid_sell"][i]
        rolling = [prepared["wholesale"][(i + k) % 8760] for k in range(24)]
        rolling_mean = fmean(rolling)
        rolling_sd = math.sqrt(sum((x - rolling_mean) ** 2 for x in rolling) / 24)
        es, ch, dis, ee = sim["ev_start"][i], sim["ev_ch"][i], sim["ev_dis"][i], sim["ev_end"][i]
        bs, bch, bdis, be = sim["bess_start"][i], sim["bess_ch"][i], sim["bess_dis"][i], sim["bess_end"][i]
        pv, load = sim["pv_total"][i], sim["load_total"][i]
        p2p_buy, p2p_sell = sim["p2p_buy"][i], sim["p2p_sell"][i]
        grid_ev_buy = max(0.0, ch - p2p_buy)
        seller = max(sell, seller_cost)
        buyer = min(buy, a["EV_WTP_EUR_kWh"])
        feasible = int(buyer >= seller)
        price = seller + a["Bargaining_Beta_Seller"] * (buyer - seller) if feasible and p2p_buy + p2p_sell > 0 else 0.0
        reserve, activation = sim["v2g_reserve"][i], sim["v2g_activation"][i]
        availability_revenue = a["V2G_Availability_Rate_EUR_kW_h"] * reserve
        activation_revenue = a["V2G_Activation_Rate_EUR_kWh"] * activation
        ev_deg = a["EV_Degradation_EUR_kWh"] * (ch + dis)
        inconvenience = (
            0.5 * a["EV_Inconvenience_Zeta_EUR_kWh2"]
            * max(0.0, a["EV_Preferred_Energy_kWh"] - ee) ** 2 * sim["connected"][i]
        )
        participation = a["Participation_Cost_EUR_Session"] * sim["session_start"][i] if mode > 0 else 0.0
        ev_benefit = (
            a["EV_Mobility_Value_EUR_km"] * sim["distance"][i]
            + p2p_sell * price + availability_revenue + activation_revenue
            - p2p_buy * price - grid_ev_buy * buy - ev_deg - inconvenience - participation
        )
        q = sim["grid_import"][i]
        network = (
            a["Network_Linear_Cost_EUR_kWh"] * q
            + 0.5 * a["Network_Quadratic_Cost_EUR_kW2h"] * q ** 2
            + 0.5 * a["Network_Congestion_Penalty_EUR_kW2h"]
            * max(0.0, q - a["Community_Grid_Capacity_kW"]) ** 2
        )
        marginal = (
            a["Network_Linear_Cost_EUR_kWh"]
            + a["Network_Quadratic_Cost_EUR_kW2h"] * q
            + a["Network_Congestion_Penalty_EUR_kW2h"] * max(0.0, q - a["Community_Grid_Capacity_kW"])
        )
        co2 = a["Emission_Factor_kg_kWh"] * q
        carbon = a["Carbon_Price_EUR_t"] * co2 / 1000
        prosumer = (
            a["Building_Energy_Utility_EUR_kWh"] * load + p2p_buy * price - p2p_sell * price
            + max(0.0, sim["grid_export"][i] - activation) * sell
            - max(0.0, q - grid_ev_buy) * buy - bess_deg * (bch + bdis)
            - a["PV_Operating_Cost_EUR_kWh"] * pv
        )
        dso = a["Network_Charge_EUR_kWh"] * (q + sim["grid_export"][i]) - availability_revenue - activation_revenue - network
        real_cost = (
            buy * q - sell * sim["grid_export"][i] + ev_deg + bess_deg * (bch + bdis)
            + a["PV_Operating_Cost_EUR_kWh"] * pv + inconvenience + network + carbon
        )
        welfare = a["EV_Mobility_Value_EUR_km"] * sim["distance"][i] + a["Building_Energy_Utility_EUR_kWh"] * load - real_cost
        ev_bounds = "PASS" if (
            ee >= a["EV_Min_Energy_kWh"] - 1e-6 and ee <= a["EV_Max_Energy_kWh"] + 1e-6
            and ch <= sim["connected"][i] * a["EV_Power_kW"] + 1e-6
            and dis <= sim["connected"][i] * a["EV_Power_kW"] + 1e-6
        ) else "FAIL"
        bess_bounds = "PASS" if (
            be >= a["Community_BESS_Min_Energy_kWh"] - 1e-6
            and be <= a["Community_BESS_Max_Energy_kWh"] + 1e-6
            and bch <= a["Community_BESS_Power_kW"] + 1e-6
            and bdis <= a["Community_BESS_Power_kW"] + 1e-6
        ) else "FAIL"
        balance = pv + q - sim["grid_export"][i] + bdis - bch + dis - ch - load
        desired = min(a["EV_Power_kW"], max(0.0, a["EV_Preferred_Energy_kWh"] - es) / a["EV_Charge_Efficiency"])
        row = [
            f"{case_id}_m{mode}", prepared["timestamp"][i], prepared["month"][i], prepared["hour"][i],
            wh, buy, sell, pv, load, sim["connected"][i], sim["location"][i], sim["distance"][i],
            sim["trip_energy"][i], sim["departure"][i],
            a["Next_Trip_Energy_kWh"] + a["EV_Buffer_kWh"] if sim["departure"][i] else 0.0,
            es, ch, dis, ee, ev_bounds, bs, bch, bdis, be, bess_bounds, sim["pv_self"][i], sim["surplus"][i],
            sim["deficit"][i], rolling_mean, rolling_sd,
            rolling_mean - a["Price_Threshold_Gamma"] * rolling_sd,
            rolling_mean + a["Price_Threshold_Gamma"] * rolling_sd,
            desired, p2p_buy, grid_ev_buy, p2p_sell, reserve, activation,
            "FAIL" if ch > 1e-6 and dis > 1e-6 else "PASS", mode,
            pv + bdis + dis - load - bch - ch, q, sim["grid_export"][i], balance,
            seller, buyer, price, feasible, max(0.0, (buyer - seller) * (p2p_buy + p2p_sell)),
            availability_revenue, activation_revenue,
            availability_revenue + activation_revenue - ev_deg - inconvenience - participation
            - a["Required_Participation_Premium_EUR_Session"] * sim["session_start"][i],
            ev_deg, inconvenience, participation, ev_benefit, prosumer, dso, co2, carbon, real_cost, welfare,
            0.0, 0.0, 0.0, (ch + dis) / (2 * a["EV_Capacity_kWh"]), 0,
            network, marginal, base_network[i] + base_carbon[i] - network - carbon,
        ]
        rows.append(row)
        ev_benefits.append(ev_benefit)

    ir = 1 if mode == 0 else int(total(ev_benefits) >= a["Strongest_Outside_Option_EV_Benefit_EUR"] - 1e-6)
    for row in rows:
        row[66] = ir
    return rows


def scenario_summary(rows: list[list[Any]], ev_capacity: float) -> dict[str, float]:
    idx = {h: i for i, h in enumerate(HEADERS)}
    def s(header: str) -> float:
        return total(row[idx[header]] for row in rows)
    p2p_volume = s("P2P_EV_Buy_kWh [M-EQ18]") + s("P2P_EV_Sell_kWh [M-EQ20]")
    p2p_value = total(
        row[idx["P2P_Price_EUR_kWh [M-EQ29]"]]
        * (row[idx["P2P_EV_Buy_kWh [M-EQ18]"]] + row[idx["P2P_EV_Sell_kWh [M-EQ20]"]])
        for row in rows
    )
    pv = s("Community_PV_Generation_kWh [M-EQ02]")
    load = s("Community_Building_Load_kWh [M-EQ03]")
    return {
        "grid_import": s("Community_Grid_Import_kWh [M-EQ26]"),
        "grid_export": s("Community_Grid_Export_kWh [M-EQ26]"),
        "ev_charge": s("EV_Charge_Power_kW [DV01]"),
        "ev_discharge": s("EV_Discharge_Power_kW [DV02]"),
        "p2p_buy": s("P2P_EV_Buy_kWh [M-EQ18]"),
        "p2p_sell": s("P2P_EV_Sell_kWh [M-EQ20]"),
        "reserve": s("V2G_Reserve_kW [M-EQ21]"),
        "activation": s("V2G_Activated_kWh [M-EQ22]"),
        "ev_benefit": s("EV_Net_Benefit_EUR [M-EQ38]"),
        "prosumer_benefit": s("Prosumer_Net_Benefit_EUR [M-EQ39]"),
        "dso_benefit": s("DSO_Net_Benefit_EUR [M-EQ40]"),
        "real_cost": s("Community_Real_System_Cost_EUR [M-EQ43]"),
        "welfare": s("Community_Social_Welfare_EUR [M-EQ44]"),
        "co2": s("Grid_Related_CO2_kg [M-EQ41]"),
        "carbon": s("Carbon_Externality_Cost_EUR [M-EQ42]"),
        "network": s("Convex_Network_Cost_EUR [M-EQ63]"),
        "avoided": s("Avoided_Production_Externality_EUR [M-EQ65]"),
        "efc": (s("EV_Charge_Power_kW [DV01]") + s("EV_Discharge_Power_kW [DV02]")) / (2 * ev_capacity),
        "scr": s("PV_Self_Consumption_kWh [M-EQ13]") / pv if pv else 0.0,
        "ssr": min(1.0, (s("PV_Self_Consumption_kWh [M-EQ13]") + s("P2P_EV_Sell_kWh [M-EQ20]")) / load) if load else 0.0,
        "p2p_vwap": p2p_value / p2p_volume if p2p_volume else 0.0,
        "ir": int(rows[0][66]),
        "max_balance": max(abs(float(row[43])) for row in rows),
    }


def formula_templates(a_ref: dict[str, str], case_id: str) -> dict[int, str]:
    a = a_ref
    formulas = {
        2: "=MONTH(B5)", 4: "='Raw_Input_2025'!J5/1000",
        5: f"=MAX(E5+{a['Retail_Adder_EUR_kWh']},{a['Minimum_Grid_Buy_EUR_kWh']})",
        6: f"=MAX({a['Grid_Export_Multiplier']}*E5,{a['Minimum_Grid_Sell_EUR_kWh']})",
        7: "='Raw_Input_2025'!M5", 8: "='Raw_Input_2025'!N5",
        12: f"=L5*{a['EV_Consumption_kWh_km']}",
        14: f"=IF(N5=1,{a['Next_Trip_Energy_kWh']}+{a['EV_Buffer_kWh']},0)",
        18: f"=P5+{a['EV_Charge_Efficiency']}*Q5-R5/{a['EV_Discharge_Efficiency']}-M5",
        23: f"=U5+{a['BESS_Charge_Efficiency']}*V5-W5/{a['BESS_Discharge_Efficiency']}",
        25: "=MIN(H5,I5+V5+Q5)", 26: "=MAX(0,H5+W5-I5-V5)", 27: "=MAX(0,I5+V5-H5-W5)",
        28: "=AVERAGE(E5:E28)", 29: "=STDEV.P(E5:E28)",
        30: f"=AC5-{a['Price_Threshold_Gamma']}*AD5", 31: f"=AC5+{a['Price_Threshold_Gamma']}*AD5",
        32: f"=MIN({a['EV_Power_kW']},MAX(0,{a['EV_Preferred_Energy_kWh']}-P5)/{a['EV_Charge_Efficiency']})",
        34: "=MAX(0,Q5-AH5)", 38: '=IF(AND(Q5>0.000001,R5>0.000001),"FAIL","PASS")',
        40: "=H5+W5+R5-I5-V5-Q5", 43: "=H5+AP5-AQ5+W5-V5+R5-Q5-I5",
        44: f"=MAX(G5,{a['PV_Operating_Cost_EUR_kWh']}+MAX({a['Residential_BESS_Degradation_EUR_kWh']},{a['Commercial_BESS_Degradation_EUR_kWh']}))",
        45: f"=MIN(F5,{a['EV_WTP_EUR_kWh']})", 46: f"=IF(AND(AV5=1,AH5+AJ5>0),AS5+{a['Bargaining_Beta_Seller']}*(AT5-AS5),0)",
        47: "=--(AT5>=AS5)", 48: "=MAX(0,(AT5-AS5)*(AH5+AJ5))",
        49: f"={a['V2G_Availability_Rate_EUR_kW_h']}*AK5", 50: f"={a['V2G_Activation_Rate_EUR_kWh']}*AL5",
        52: f"={a['EV_Degradation_EUR_kWh']}*(Q5+R5)",
        53: f"=0.5*{a['EV_Inconvenience_Zeta_EUR_kWh2']}*MAX(0,{a['EV_Preferred_Energy_kWh']}-S5)^2*J5",
        54: f"=IF(AND(AN5>0,'Raw_Input_2025'!O5=1),{a['Participation_Cost_EUR_Session']},0)",
        55: "=MobilityUtility+P2PRevenue+V2GRevenue-Charging-P2PBuy-Degradation-Inconvenience-Participation",
        56: "=EnergyUtility+P2PNet+GridExport-AvoidedGridPurchase-BESSDegradation-PVOperatingCost",
        57: "=NetworkRevenue-V2GAvailability-V2GActivation-NetworkCost", 58: f"={a['Emission_Factor_kg_kWh']}*AP5",
        59: f"={a['Carbon_Price_EUR_t']}*BG5/1000", 60: "=GridEnergy+Degradation+Operating+Inconvenience+Network+Carbon",
        61: "=MobilityUtility+EnergyUtility-BI5", 62: "=InternalP2PPayments+InternalV2GPayments=0",
        63: "=0", 64: f"={a['VOLL_EUR_kWh']}*BL5", 65: f"=(Q5+R5)/(2*{a['EV_Capacity_kWh']})",
        66: "='Annual_Summary'!X5",
        67: f"={a['Network_Linear_Cost_EUR_kWh']}*AP5+0.5*{a['Network_Quadratic_Cost_EUR_kW2h']}*AP5^2+0.5*{a['Network_Congestion_Penalty_EUR_kW2h']}*MAX(0,AP5-{a['Community_Grid_Capacity_kW']})^2",
        68: f"={a['Network_Linear_Cost_EUR_kWh']}+{a['Network_Quadratic_Cost_EUR_kW2h']}*AP5+{a['Network_Congestion_Penalty_EUR_kW2h']}*MAX(0,AP5-{a['Community_Grid_Capacity_kW']})",
        69: "='Raw_Input_2025'!K5+'Raw_Input_2025'!L5-BP5-BH5",
    }
    formulas[9] = '=--(EV is physically connected according to schedule)'
    formulas[10] = '=IF(travel,"Travel",IF(daytime,"Commercial","Residential"))'
    formulas[11] = '=IF(travel block,scheduled distance,0)'
    formulas[13] = '=--(hour is a scheduled departure)'
    formulas[19] = '=IF(AND(EV energy/power bounds),"PASS","FAIL")'
    formulas[24] = '=IF(AND(BESS energy/power bounds),"PASS","FAIL")'
    return formulas


class BookFormats:
    def __init__(self, workbook: xlsxwriter.Workbook):
        self.title = workbook.add_format({"bold": True, "font_color": COLORS["white"], "bg_color": COLORS["navy"], "font_size": 15, "valign": "vcenter"})
        self.subtitle = workbook.add_format({"italic": True, "font_color": COLORS["navy"], "bg_color": COLORS["blue"], "text_wrap": True, "valign": "vcenter"})
        self.header = workbook.add_format({"bold": True, "font_color": COLORS["white"], "bg_color": COLORS["teal"], "border": 1, "border_color": COLORS["line"], "text_wrap": True, "align": "center", "valign": "vcenter"})
        self.header_navy = workbook.add_format({"bold": True, "font_color": COLORS["white"], "bg_color": COLORS["navy"], "border": 1, "text_wrap": True, "align": "center", "valign": "vcenter"})
        self.header_formula = workbook.add_format({"bold": True, "font_color": COLORS["navy"], "bg_color": COLORS["formula"], "border": 1, "text_wrap": True, "align": "center", "valign": "vcenter"})
        self.header_decision = workbook.add_format({"bold": True, "font_color": COLORS["navy"], "bg_color": COLORS["decision"], "border": 1, "text_wrap": True, "align": "center", "valign": "vcenter"})
        self.label = workbook.add_format({"bold": True, "font_color": COLORS["navy"], "bg_color": COLORS["pale"], "border": 1})
        self.input = workbook.add_format({"bg_color": COLORS["input"], "num_format": "0.0000", "border": 1})
        self.text = workbook.add_format({"text_wrap": True, "valign": "top"})
        self.text_border = workbook.add_format({"text_wrap": True, "valign": "top", "border": 1})
        self.num4 = workbook.add_format({"num_format": "0.0000"})
        self.num2 = workbook.add_format({"num_format": "#,##0.00"})
        self.integer = workbook.add_format({"num_format": "0"})
        self.percent = workbook.add_format({"num_format": "0.00%"})
        self.datetime = workbook.add_format({"num_format": "yyyy-mm-dd hh:mm"})
        self.date = workbook.add_format({"num_format": "yyyy-mm-dd"})
        self.pass_fmt = workbook.add_format({"bg_color": COLORS["good"]})
        self.fail_fmt = workbook.add_format({"bg_color": COLORS["bad"]})


def title(ws, fm: BookFormats, text: str, subtitle: str, last_col: int) -> None:
    ws.merge_range(0, 0, 0, last_col, text, fm.title)
    ws.merge_range(1, 0, 1, last_col, subtitle, fm.subtitle)
    ws.set_row(0, 28)
    ws.set_row(1, 32)
    ws.hide_gridlines(2)


def write_headers(ws, fm: BookFormats, row: int, headers: list[str], special: bool = False) -> None:
    for col, value in enumerate(headers):
        fmt = fm.header
        if special:
            fmt = fm.header_decision if col in DECISION_COLS else fm.header_formula if col in FORMULA_COLS else fm.header
        ws.write(row, col, value, fmt)
    ws.set_row(row, 40)


def write_value(ws, row: int, col: int, value: Any, fmt=None) -> None:
    if value is None:
        ws.write_blank(row, col, None, fmt)
    elif isinstance(value, bool):
        ws.write_boolean(row, col, value, fmt)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        ws.write_number(row, col, float(value), fmt)
    else:
        ws.write_string(row, col, str(value), fmt)


def add_readme(wb, fm, model, case_id, mode, filename):
    ws = wb.add_worksheet("README")
    title(ws, fm, f"{case_id} mode m={mode}: 8,760-hour P2P–V2G output", "Colab-compatible formula-audit research workbook.", 3)
    case = model["cases"][case_id]
    rows = [
        ("File", filename),
        ("Use case", "Private EV between residential and commercial prosumers" if case_id == "UC1" else "EV taxi with commercial parking prosumer"),
        ("EV mode", f"m={mode}: {mode_description(mode)}"),
        ("Time structure", "8,760 hourly outputs; 24-hour rolling optimisation; first-hour action retained"),
        ("Prosumer configuration", "; ".join(f"{n['name']} Strategy {n['strategy']}, BESS {n['bess_capacity']} kWh" for n in case["nodes"])),
        ("Equations", "M-EQ01–M-EQ62 plus M-EQ63, M-EQ64 and M-EQ65"),
        ("Colour legend", "Yellow = assumption; orange = optimiser decision; green = equation-derived output"),
        ("Recalculation", "Change assumptions in source code and rerun the optimiser; dispatch decisions are not recalculated by Excel alone."),
        ("Input source", "Input Data(8).xlsx: 2025 prices, PV and base-load profile"),
        ("Runtime", "Google Colab-compatible Python implementation using SciPy, openpyxl and XlsxWriter"),
    ]
    for r, (label, value) in enumerate(rows, 3):
        ws.write(r, 0, label, fm.label)
        ws.write(r, 1, value, fm.text_border)
    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 94)
    ws.set_column(2, 3, 4)


def add_assumptions(wb, fm, assumptions):
    ws = wb.add_worksheet("Assumptions")
    title(ws, fm, "Editable assumptions and model controls", "Yellow cells are input assumptions. Rerun the model after changing dispatch-driving parameters.", 4)
    write_headers(ws, fm, 3, ["Parameter", "Category", "Value", "Unit / role", "Source / status"])
    refs = {}
    for i, (key, value) in enumerate(assumptions.items(), 4):
        category = "EV/mobility" if key.startswith("EV_") or "Trip" in key else "Prosumer" if "BESS" in key or "PV" in key or "Load" in key else "Economic/model"
        ws.write(i, 0, key)
        ws.write(i, 1, category)
        write_value(ws, i, 2, value, fm.input if isinstance(value, (int, float)) else fm.text_border)
        ws.write(i, 3, "See parameter name")
        ws.write(i, 4, "Calculated IR reference" if key.startswith("Outside_") or key.startswith("Strongest_") else "Editable modelling assumption")
        refs[key] = f"'Assumptions'!$C${i + 1}"
    ws.set_column(0, 0, 40)
    ws.set_column(1, 1, 18)
    ws.set_column(2, 2, 25)
    ws.set_column(3, 3, 20)
    ws.set_column(4, 4, 42)
    ws.freeze_panes(4, 0)
    return refs


def add_raw_input(wb, fm, model, case_id, mode, assumptions):
    ws = wb.add_worksheet("Raw_Input_2025")
    title(ws, fm, "Raw and processed hourly inputs", "A:J reproduce the supplied source; K:O contain baseline and aligned support inputs.", 14)
    headers = list(model["source_header"]) + ["Baseline_Network_Cost_EUR", "Baseline_Carbon_Cost_EUR", "Community_PV_kWh [M-EQ02]", "Community_Load_kWh [M-EQ03]", "EV_Session_Start_Flag"]
    write_headers(ws, fm, 3, headers)
    network, carbon = baseline_externality(model, case_id, assumptions)
    case, prepared = model["cases"][case_id], model["prepared_inputs"]
    sim = case["modes"][str(mode)]
    for i, source_row in enumerate(model["source_rows"], 4):
        row_idx = i - 4
        extra = [
            network[row_idx], carbon[row_idx],
            prepared["residential_pv"][row_idx] + prepared["commercial_pv"][row_idx] if case_id == "UC1" else prepared["commercial_pv"][row_idx],
            prepared["residential_load"][row_idx] + prepared["commercial_load"][row_idx] if case_id == "UC1" else prepared["commercial_load"][row_idx],
            sim["session_start"][row_idx],
        ]
        for col, value in enumerate(list(source_row) + extra):
            fmt = fm.datetime if col in (0, 1) else fm.num4 if col >= 6 else None
            write_value(ws, i, col, value, fmt)
    ws.set_column(0, 1, 20)
    ws.set_column(2, 5, 14)
    ws.set_column(6, 14, 23)
    ws.freeze_panes(4, 2)
    ws.autofilter(3, 0, 8763, 14)


def add_hourly(wb, fm, case_id, mode, rows):
    ws = wb.add_worksheet("Hourly_Output")
    title(ws, fm, f"{case_id} m={mode} hourly results", "Orange headers are rolling-optimiser decisions; green headers are equation-derived outputs. One row equals one hour of 2025.", len(HEADERS) - 1)
    write_headers(ws, fm, 3, HEADERS, special=True)
    for r, values in enumerate(rows, 4):
        for c, value in enumerate(values):
            if c == 1:
                fmt = fm.datetime
            elif c in (2, 3, 9, 13, 39, 47, 66):
                fmt = fm.integer
            elif c in (0, 10, 19, 24, 38):
                fmt = None
            else:
                fmt = fm.num4
            write_value(ws, r, c, value, fmt)
    ws.set_column(0, 0, 15)
    ws.set_column(1, 1, 20)
    ws.set_column(2, 3, 12)
    ws.set_column(4, len(HEADERS) - 1, 18)
    ws.set_column(10, 10, 16)
    ws.freeze_panes(4, 4)
    ws.autofilter(3, 0, 8763, len(HEADERS) - 1)


def add_formula_audit(wb, fm, a_ref, case_id):
    ws = wb.add_worksheet("Formula_Audit")
    title(ws, fm, "Excel formula specification and audit trail", "Hourly decisions are evaluated by the rolling model; this sheet preserves the Excel-equivalent formula template for every derived output.", 5)
    write_headers(ws, fm, 3, ["Column", "Output header", "Equation ID", "Cell type", "Excel formula template", "Implementation note"])
    formulas = formula_templates(a_ref, case_id)
    for i, header in enumerate(HEADERS, 4):
        col = i - 4
        letter = xl_col_to_name(col)
        equation = ", ".join(part for part in header.replace("]", "[").split("[") if part.startswith("M-EQ"))
        if col in DECISION_COLS:
            cell_type, note = "Rolling optimiser output", "Rerun optimiser after changing dispatch-driving assumptions"
        elif col in formulas:
            cell_type, note = "Equation-derived output", "Excel-equivalent audit template"
        else:
            cell_type, note = "Source/input", "Direct source value or identifier"
        ws.write(i, 0, letter)
        ws.write(i, 1, header)
        ws.write(i, 2, equation)
        ws.write(i, 3, cell_type)
        ws.write_string(i, 4, formulas.get(col, ""), fm.text)
        ws.write(i, 5, note, fm.text)
    extras = [
        ("Cooperative_Game", "M-EQ46–M-EQ52", "Characteristic function, fuzzy value, Shapley, tau, core and AL"),
        ("Resilience", "M-EQ53–M-EQ55", "EENS, outage cost and resilience value"),
        ("Monthly/Annual summaries", "M-EQ57–M-EQ60, M-EQ62", "Ratio-of-sums, VWAP, IR and temporal aggregation"),
        ("Price_Transmission", "M-EQ61", "Covariance/variance pass-through diagnostic"),
    ]
    row = 4 + len(HEADERS)
    for name, eq, note in extras:
        ws.write_row(row, 0, ["—", name, eq, "Post-processing/formulas", "See the named sheet", note])
        row += 1
    ws.set_column(0, 0, 11)
    ws.set_column(1, 1, 50)
    ws.set_column(2, 3, 24)
    ws.set_column(4, 4, 76)
    ws.set_column(5, 5, 48)
    ws.freeze_panes(4, 0)


def aggregate_month(rows, month):
    selected = [r for r in rows if int(r[2]) == month]
    indices = [41, 42, 16, 17, 33, 35, 36, 37, 55, 56, 57, 60, 61, 58, 59, 67, 69]
    values = [total(r[i] for r in selected) for i in indices]
    pv = total(r[7] for r in selected)
    load = total(r[8] for r in selected)
    scr = total(r[25] for r in selected) / pv if pv else 0.0
    ssr = min(1.0, (total(r[25] for r in selected) + total(r[35] for r in selected)) / load) if load else 0.0
    volume = total(r[33] + r[35] for r in selected)
    vwap = total(r[46] * (r[33] + r[35]) for r in selected) / volume if volume else 0.0
    return values + [scr, ssr, vwap]


def add_monthly(wb, fm, rows):
    ws = wb.add_worksheet("Monthly_Summary")
    title(ws, fm, "Monthly results", "Aggregations are formula-linked to Hourly_Output; cached values permit immediate viewing in Colab.", 21)
    headers = ["Month_ID", "Month", "Grid_Import_kWh", "Grid_Export_kWh", "EV_Charge_kWh", "EV_Discharge_kWh", "P2P_Buy_kWh", "P2P_Sell_kWh", "V2G_Reserve_kW-h", "V2G_Activation_kWh", "EV_Net_Benefit_EUR", "Prosumer_Net_Benefit_EUR", "DSO_Net_Benefit_EUR", "Real_System_Cost_EUR", "Social_Welfare_EUR", "CO2_kg", "Carbon_Cost_EUR", "Network_Cost_EUR", "Avoided_Externality_EUR", "SCR [M-EQ57]", "SSR [M-EQ58]", "P2P_VWAP_EUR_kWh [M-EQ59]"]
    write_headers(ws, fm, 3, headers)
    names = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
    src_cols = ["AP", "AQ", "Q", "R", "AH", "AJ", "AK", "AL", "BD", "BE", "BF", "BI", "BJ", "BG", "BH", "BP", "BR"]
    for month in range(1, 13):
        row = 3 + month
        calc = aggregate_month(rows, month)
        ws.write(row, 0, month)
        ws.write(row, 1, names[month - 1])
        for offset, (source_col, cached) in enumerate(zip(src_cols, calc[:17]), 2):
            formula = f'=SUMIFS(\'Hourly_Output\'!${source_col}$5:${source_col}$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1})'
            ws.write_formula(row, offset, formula, fm.num2, cached)
        ws.write_formula(row, 19, f'=IFERROR(SUMIFS(\'Hourly_Output\'!$Z$5:$Z$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1})/SUMIFS(\'Hourly_Output\'!$H$5:$H$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1}),0)', fm.percent, calc[17])
        ws.write_formula(row, 20, f'=IFERROR(MIN(1,(SUMIFS(\'Hourly_Output\'!$Z$5:$Z$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1})+SUMIFS(\'Hourly_Output\'!$AJ$5:$AJ$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1}))/SUMIFS(\'Hourly_Output\'!$I$5:$I$8764,\'Hourly_Output\'!$C$5:$C$8764,$A{row + 1})),0)', fm.percent, calc[18])
        ws.write_formula(row, 21, f'=IFERROR(SUMPRODUCT((\'Hourly_Output\'!$C$5:$C$8764=$A{row + 1})*\'Hourly_Output\'!$AU$5:$AU$8764*(\'Hourly_Output\'!$AH$5:$AH$8764+\'Hourly_Output\'!$AJ$5:$AJ$8764))/(G{row + 1}+H{row + 1}),0)', fm.num4, calc[19])
    ws.set_column(0, 0, 12)
    ws.set_column(1, 1, 17)
    ws.set_column(2, 21, 19)
    ws.freeze_panes(4, 2)


def add_annual(wb, fm, mode, rows, summary, assumptions):
    ws = wb.add_worksheet("Annual_Summary")
    title(ws, fm, "Annual results and primary KPIs", "Formula-linked totals for 2025; IR compares the EV result with the strongest disclosed outside option.", 24)
    headers = ["Mode", "Description", "Grid_Import_kWh", "Grid_Export_kWh", "EV_Charge_kWh", "EV_Discharge_kWh", "P2P_Buy_kWh", "P2P_Sell_kWh", "V2G_Reserve_kW-h", "V2G_Activation_kWh", "EV_Net_Benefit_EUR", "Prosumer_Net_Benefit_EUR", "DSO_Net_Benefit_EUR", "Real_System_Cost_EUR", "Social_Welfare_EUR", "CO2_kg", "Carbon_Cost_EUR", "Network_Cost_EUR", "Avoided_Externality_EUR", "EV_EFC", "SCR", "SSR", "P2P_VWAP_EUR_kWh", "IR_Flag [M-EQ60]", "Max_Balance_Residual_kWh"]
    write_headers(ws, fm, 3, headers)
    ws.write(4, 0, mode)
    ws.write(4, 1, mode_description(mode))
    keys = ["grid_import", "grid_export", "ev_charge", "ev_discharge", "p2p_buy", "p2p_sell", "reserve", "activation", "ev_benefit", "prosumer_benefit", "dso_benefit", "real_cost", "welfare", "co2", "carbon", "network", "avoided", "efc"]
    src_cols = ["AP", "AQ", "Q", "R", "AH", "AJ", "AK", "AL", "BD", "BE", "BF", "BI", "BJ", "BG", "BH", "BP", "BR", "BN"]
    for col, (key, src) in enumerate(zip(keys, src_cols), 2):
        ws.write_formula(4, col, f"=SUM('Hourly_Output'!${src}$5:${src}$8764)", fm.num2, summary[key])
    ws.write_formula(4, 20, "=IFERROR(SUM('Hourly_Output'!$Z$5:$Z$8764)/SUM('Hourly_Output'!$H$5:$H$8764),0)", fm.percent, summary["scr"])
    ws.write_formula(4, 21, "=IFERROR(MIN(1,(SUM('Hourly_Output'!$Z$5:$Z$8764)+SUM('Hourly_Output'!$AJ$5:$AJ$8764))/SUM('Hourly_Output'!$I$5:$I$8764)),0)", fm.percent, summary["ssr"])
    ws.write_formula(4, 22, "=IFERROR(SUMPRODUCT('Hourly_Output'!$AU$5:$AU$8764,('Hourly_Output'!$AH$5:$AH$8764+'Hourly_Output'!$AJ$5:$AJ$8764))/(G5+H5),0)", fm.num4, summary["p2p_vwap"])
    ws.write_formula(4, 23, f'=IF(A5=0,1,--(K5>=\'Assumptions\'!$C${list(assumptions).index("Strongest_Outside_Option_EV_Benefit_EUR") + 5}-0.000001))', fm.integer, summary["ir"])
    ws.write_formula(4, 24, "=MAX(MAX('Hourly_Output'!$AR$5:$AR$8764),-MIN('Hourly_Output'!$AR$5:$AR$8764))", fm.num4, summary["max_balance"])
    ws.set_column(0, 0, 10)
    ws.set_column(1, 1, 43)
    ws.set_column(2, 24, 19)


def coalition_cost_parts(case, key):
    base = case["no_der_modes"]["0"]["cost_summary"]
    maps = {
        "empty": base, "P": case["modes"]["0"]["cost_summary"], "E": case["no_der_modes"]["1"]["cost_summary"], "D": base,
        "EP": case["modes"]["1"]["cost_summary"], "DP": case["modes"]["0"]["cost_summary"],
        "DE": case["no_der_modes"]["2"]["cost_summary"], "DEP": case["modes"]["2"]["cost_summary"],
    }
    return base, maps[key]


def add_cooperative(wb, fm, case, assumptions):
    ws = wb.add_worksheet("Cooperative_Game")
    title(ws, fm, "Cooperative-game values and allocations", "General P–E–D game and DSO-clan transformation; convexity, core and tau applicability are tested.", 10)
    write_headers(ws, fm, 3, ["Coalition", "Members", "Baseline non-carbon cost", "Coalition non-carbon cost", "RV", "Carbon benefit", "v(S) [M-EQ46]", "Fuzzy μ", "Fuzzy value [M-EQ47]", "Interpretation"])
    order = ["empty", "P", "E", "D", "EP", "DP", "DE", "DEP"]
    labels = {"empty": "Empty", "P": "Prosumer", "E": "EV", "D": "DSO", "EP": "Prosumer + EV", "DP": "Prosumer + DSO", "DE": "EV + DSO", "DEP": "Grand coalition"}
    mu = assumptions["Fuzzy_Membership_Mu"]
    for i, key in enumerate(order, 4):
        base, coop = coalition_cost_parts(case, key)
        v = case["coalition"]["values"][key]
        data = [key, labels[key], base["noncarbon"], coop["noncarbon"], 0, base["carbon"] - coop["carbon"]]
        for c, value in enumerate(data):
            write_value(ws, i, c, value, fm.num2 if c >= 2 else None)
        ws.write_formula(i, 6, f"=C{i + 1}-D{i + 1}+E{i + 1}+F{i + 1}", fm.num2, v)
        ws.write(i, 7, mu, fm.num4)
        ws.write_formula(i, 8, f"=G{i + 1}*H{i + 1}", fm.num2, v * mu)
        ws.write(i, 9, "Common non-cooperative reference" if key == "empty" else "Counterfactual rolling optimisation")
    alloc_row = 14
    write_headers(ws, fm, alloc_row, ["Player", "Shapley [M-EQ48]", "Utopia [M-EQ49]", "Minimum right [M-EQ49]", "Tau [M-EQ50]", "AL/Alexia [M-EQ52]", "Core [M-EQ51]", "Convex", "Clan Shapley", "Clan Tau", "Clan AL"])
    q, clan = case["coalition"], case["coalition"]["clan"]
    for offset, player in enumerate(("P", "E", "D"), alloc_row + 1):
        ws.write(offset, 0, player)
        ws.write(offset, 1, q["shapley"][player], fm.num2)
        ws.write(offset, 2, q["tau"]["utopia"][player], fm.num2)
        ws.write(offset, 3, q["tau"]["minimum_rights"][player], fm.num2)
        ws.write(offset, 4, q["tau"]["allocation"][player], fm.num2)
        write_value(ws, offset, 5, (q.get("al", {}).get("allocation") or {}).get(player), fm.num2)
        ws.write(offset, 6, int(q["al"]["core_nonempty"]))
        ws.write(offset, 7, int(q["convexity"]["is_convex"]))
        ws.write(offset, 8, clan["shapley"][player], fm.num2)
        ws.write(offset, 9, clan["tau"]["allocation"][player], fm.num2)
        write_value(ws, offset, 10, (clan.get("al", {}).get("allocation") or {}).get(player), fm.num2)
    ws.write(20, 0, "Method note", fm.label)
    ws.write(20, 1, "Shapley and τ values are calculated from coalition-specific rolling optimisations. AL is obtained from lexicographic core programmes in Python.", fm.text)
    ws.set_column(0, 0, 17)
    ws.set_column(1, 1, 27)
    ws.set_column(2, 10, 22)


def add_resilience(wb, fm, model, case_id, mode, rows, assumptions):
    ws = wb.add_worksheet("Resilience")
    title(ws, fm, "Resilience stress tests [M-EQ53–M-EQ55]", "Two illustrative 24-hour outages with sequential depletion of available stored energy.", 12)
    headers = ["Event", "Event_Date", "Hour", "Source_Row", "Critical_Load_kWh", "PV_Supply_kWh", "Baseline_Unserved_kWh [M-EQ53]", "Backup_Start_kWh", "Backup_Used_kWh", "Backup_End_kWh", "Coordinated_Unserved_kWh [M-EQ53]", "Activation_Cost_EUR", "Note"]
    write_headers(ws, fm, 3, headers)
    prepared = model["prepared_inputs"]
    events = [("Winter_24h", 1, 15), ("Summer_24h", 7, 15)]
    event_results = {}
    out_row = 4
    for event_id, month, day in events:
        backup_end = 0.0
        base_eens = coord_eens = activation_cost = 0.0
        for hour in range(24):
            idx = next(i for i in range(8760) if prepared["month"][i] == month and prepared["day"][i] == day and prepared["hour"][i] == hour)
            critical = rows[idx][8] * assumptions["Critical_Load_Share"]
            pv = rows[idx][7]
            unserved = max(0.0, critical - pv)
            if hour == 0:
                backup_start = max(0.0, rows[idx][20] - assumptions["Community_BESS_Min_Energy_kWh"])
                if mode == 2:
                    backup_start += max(0.0, rows[idx][15] - assumptions["EV_Min_Energy_kWh"] - assumptions["EV_Buffer_kWh"])
            else:
                backup_start = backup_end
            used = min(unserved, backup_start)
            backup_end = max(0.0, backup_start - used)
            coordinated = max(0.0, unserved - used)
            act_cost = used * max(assumptions["Commercial_BESS_Degradation_EUR_kWh"], assumptions["EV_Degradation_EUR_kWh"])
            values = [event_id, prepared["timestamp"][idx], hour, idx + 5, critical, pv, unserved, backup_start, used, backup_end, coordinated, act_cost, "Illustrative outage; critical-load share and VOLL are editable"]
            for c, value in enumerate(values):
                write_value(ws, out_row, c, value, fm.date if c == 1 else fm.num2 if 4 <= c <= 11 else None)
            base_eens += unserved
            coord_eens += coordinated
            activation_cost += act_cost
            out_row += 1
        event_results[event_id] = (base_eens, coord_eens, activation_cost)
    write_headers(ws, fm, 54, ["Event", "Baseline EENS [M-EQ53]", "Coordinated EENS [M-EQ53]", "Baseline outage cost [M-EQ54]", "Coordinated outage cost [M-EQ54]", "Activation cost", "Resilience value [M-EQ55]", "Critical load served"])
    for r, (event_id, _, _) in enumerate(events, 55):
        base, coord, act = event_results[event_id]
        base_cost = assumptions["VOLL_EUR_kWh"] * base
        coord_cost = assumptions["VOLL_EUR_kWh"] * coord
        rv = base_cost - coord_cost - act
        served = 1 - coord / base if base else 1.0
        ws.write(r, 0, event_id)
        ws.write_formula(r, 1, f'=SUMIFS($G$5:$G$52,$A$5:$A$52,A{r + 1})', fm.num2, base)
        ws.write_formula(r, 2, f'=SUMIFS($K$5:$K$52,$A$5:$A$52,A{r + 1})', fm.num2, coord)
        ws.write_formula(r, 3, f'=\'Assumptions\'!$C${list(assumptions).index("VOLL_EUR_kWh") + 5}*B{r + 1}', fm.num2, base_cost)
        ws.write_formula(r, 4, f'=\'Assumptions\'!$C${list(assumptions).index("VOLL_EUR_kWh") + 5}*C{r + 1}', fm.num2, coord_cost)
        ws.write_formula(r, 5, f'=SUMIFS($L$5:$L$52,$A$5:$A$52,A{r + 1})', fm.num2, act)
        ws.write_formula(r, 6, f'=D{r + 1}-E{r + 1}-F{r + 1}', fm.num2, rv)
        ws.write_formula(r, 7, f'=IFERROR(1-C{r + 1}/B{r + 1},1)', fm.percent, served)
    ws.set_column(0, 0, 18)
    ws.set_column(1, 11, 21)
    ws.set_column(12, 12, 48)


def add_price_transmission(wb, fm, rows, summary):
    ws = wb.add_worksheet("Price_Transmission")
    title(ws, fm, "Wholesale-to-local price transmission [M-EQ61]", "Bivariate pass-through diagnostic based on all 8,760 observations.", 4)
    write_headers(ws, fm, 3, ["Metric", "Formula / specification", "Value", "Unit", "Interpretation"])
    wholesale = [r[4] for r in rows]
    p2p = [r[46] for r in rows]
    mw, mp = fmean(wholesale), fmean(p2p)
    var = sum((x - mw) ** 2 for x in wholesale) / len(wholesale)
    cov = sum((x - mw) * (y - mp) for x, y in zip(wholesale, p2p)) / len(wholesale)
    pt = cov / var if var else 0.0
    intercept = mp - pt * mw
    sst = sum((y - mp) ** 2 for y in p2p)
    sse = sum((y - (intercept + pt * x)) ** 2 for x, y in zip(wholesale, p2p))
    r2 = 1 - sse / sst if sst else 0.0
    metrics = [
        ("Transmission coefficient PT", "Cov(P2P, wholesale) / Var(wholesale)", pt, "ratio", "Pass-through of wholesale variation", "=IFERROR(COVARIANCE.P('Hourly_Output'!$AU$5:$AU$8764,'Hourly_Output'!$E$5:$E$8764)/VAR.P('Hourly_Output'!$E$5:$E$8764),0)"),
        ("Intercept alpha0", "Mean(P2P) - PT×Mean(wholesale)", intercept, "EUR/kWh", "Local-price component", "=AVERAGE('Hourly_Output'!$AU$5:$AU$8764)-C5*AVERAGE('Hourly_Output'!$E$5:$E$8764)"),
        ("R-squared", "RSQ(P2P, wholesale)", r2, "0–1", "Simple bivariate fit", "=IFERROR(RSQ('Hourly_Output'!$AU$5:$AU$8764,'Hourly_Output'!$E$5:$E$8764),0)"),
        ("Mean wholesale price", "AVERAGE hourly wholesale", mw, "EUR/kWh", "2025 input", "=AVERAGE('Hourly_Output'!$E$5:$E$8764)"),
        ("Mean P2P price", "AVERAGE hourly P2P price", mp, "EUR/kWh", "Includes zero-volume hours", "=AVERAGE('Hourly_Output'!$AU$5:$AU$8764)"),
        ("P2P volume-weighted price", "Annual VWAP [M-EQ59]", summary["p2p_vwap"], "EUR/kWh", "Traded-price KPI", "='Annual_Summary'!W5"),
    ]
    for r, metric in enumerate(metrics, 4):
        ws.write(r, 0, metric[0])
        ws.write(r, 1, metric[1])
        ws.write_formula(r, 2, metric[5], fm.num4, metric[2])
        ws.write(r, 3, metric[3])
        ws.write(r, 4, metric[4])
    ws.write_row(10, 0, ["Extended specification", "pP2P = α0 + α1 wholesale + α2 network + α3 carbon + ε", "", "Text", "Estimate multivariate coefficients separately"])
    ws.set_column(0, 0, 31)
    ws.set_column(1, 1, 68)
    ws.set_column(2, 3, 18)
    ws.set_column(4, 4, 49)


def add_validation(wb, fm, rows, summary, assumptions):
    ws = wb.add_worksheet("Validation")
    title(ws, fm, "Validation and reproducibility checks", "Physical/accounting checks exclude IR because an evaluated mode may rationally be rejected.", 9)
    write_headers(ws, fm, 3, ["Check", "Calculated value", "Tolerance / rule", "Status", "M-EQ", "Scope", "Expected", "Observed units", "Action if FAIL", "Note"])
    checks = [
        ("Hourly row count", len(rows), 8760, len(rows) == 8760, "M-EQ62", "rows"),
        ("Maximum absolute balance residual", max(abs(r[43]) for r in rows), 1e-6, max(abs(r[43]) for r in rows) <= 1e-6, "M-EQ26", "kWh"),
        ("EV bound violations", sum(r[19] == "FAIL" for r in rows), 0, not any(r[19] == "FAIL" for r in rows), "M-EQ10", "rows"),
        ("BESS bound violations", sum(r[24] == "FAIL" for r in rows), 0, not any(r[24] == "FAIL" for r in rows), "M-EQ12", "rows"),
        ("Simultaneous EV charge/discharge", sum(r[38] == "FAIL" for r in rows), 0, not any(r[38] == "FAIL" for r in rows), "M-EQ23", "rows"),
        ("Maximum transfer-check residual", max(abs(r[62]) for r in rows), 1e-6, max(abs(r[62]) for r in rows) <= 1e-6, "M-EQ45", "EUR"),
        ("Departure preferred-SOC violations", sum(r[13] == 1 and r[15] < assumptions["EV_Preferred_Energy_kWh"] - 1e-6 for r in rows), 0, not any(r[13] == 1 and r[15] < assumptions["EV_Preferred_Energy_kWh"] - 1e-6 for r in rows), "M-EQ08", "departures"),
        ("Annual individual rationality", summary["ir"], 1, summary["ir"] == 1, "M-EQ60", "flag"),
    ]
    for r, check in enumerate(checks, 4):
        ws.write(r, 0, check[0])
        write_value(ws, r, 1, check[1], fm.num4)
        write_value(ws, r, 2, check[2], fm.num4)
        ws.write(r, 3, "PASS" if check[3] else "FAIL", fm.pass_fmt if check[3] else fm.fail_fmt)
        ws.write(r, 4, check[4])
        ws.write(r, 5, "Annual" if r in (4, 11) else "Hourly")
        ws.write(r, 6, "See tolerance")
        ws.write(r, 7, check[5])
        ws.write(r, 8, "Inspect source inputs and optimiser decisions")
        ws.write(r, 9, "IR failure is an economic result, not a physical-model error" if check[4] == "M-EQ60" else "")
    physical_pass = all(check[3] for check in checks[:-1])
    write_headers(ws, fm, 13, ["Overall physical/accounting status", "Result"])
    ws.write(14, 0, "Excludes IR")
    ws.write(14, 1, "PASS" if physical_pass else "FAIL", fm.pass_fmt if physical_pass else fm.fail_fmt)
    ws.set_column(0, 0, 39)
    ws.set_column(1, 3, 20)
    ws.set_column(4, 8, 23)
    ws.set_column(9, 9, 43)


def add_equation_map(wb, fm, model):
    ws = wb.add_worksheet("Equation_Map")
    title(ws, fm, "Numbered methodology equation map", "M-EQ01–M-EQ62 are preserved from the specification; M-EQ63–M-EQ65 are the three production-externality additions.", 5)
    write_headers(ws, fm, 3, ["Equation ID", "Title", "Equation", "Purpose", "Excel / code implementation", "Literature basis"])
    for r, equation in enumerate(model["equations"], 4):
        values = [equation["id"], equation["title"], equation["formula"], equation["purpose"], equation["implementation"], equation["literature"]]
        for c, value in enumerate(values):
            ws.write(r, c, value, fm.text)
    ws.set_column(0, 0, 13)
    ws.set_column(1, 1, 36)
    ws.set_column(2, 2, 54)
    ws.set_column(3, 5, 54)
    ws.freeze_panes(4, 0)


def build_mode_workbook(model, output_dir: Path, case_id: str, mode: int) -> Path:
    label = "Private_EV" if case_id == "UC1" else "EV_Taxi"
    filename = f"{case_id}_m{mode}_{label}_8760_M_EQ01_65.xlsx"
    output = output_dir / filename
    assumptions = enrich_assumptions(model, case_id, mode)
    rows = hourly_matrix(model, case_id, mode, assumptions)
    summary = scenario_summary(rows, model["cases"][case_id]["ev"]["capacity"])
    wb = xlsxwriter.Workbook(output, {"constant_memory": True, "nan_inf_to_errors": True})
    wb.set_properties({"title": filename, "subject": "Mobility-aware P2P–V2G local-market model", "author": "Stathis Devves"})
    wb.set_calc_mode("auto")
    fm = BookFormats(wb)
    add_readme(wb, fm, model, case_id, mode, filename)
    refs = add_assumptions(wb, fm, assumptions)
    add_raw_input(wb, fm, model, case_id, mode, assumptions)
    add_hourly(wb, fm, case_id, mode, rows)
    add_formula_audit(wb, fm, refs, case_id)
    add_monthly(wb, fm, rows)
    add_annual(wb, fm, mode, rows, summary, assumptions)
    add_cooperative(wb, fm, model["cases"][case_id], assumptions)
    add_resilience(wb, fm, model, case_id, mode, rows, assumptions)
    add_price_transmission(wb, fm, rows, summary)
    add_validation(wb, fm, rows, summary, assumptions)
    add_equation_map(wb, fm, model)
    wb.close()
    print(f"Exported and cached formulas: {output}", flush=True)
    return output


def build_comparative(model, output_dir: Path) -> Path:
    output = output_dir / "Comparative_Summary_UC1_UC2_M_EQ01_65.xlsx"
    all_rows = []
    for case_id in ("UC1", "UC2"):
        for mode in (0, 1, 2):
            assumptions = enrich_assumptions(model, case_id, mode)
            rows = hourly_matrix(model, case_id, mode, assumptions)
            all_rows.append((case_id, mode, scenario_summary(rows, model["cases"][case_id]["ev"]["capacity"])))
    wb = xlsxwriter.Workbook(output, {"constant_memory": True})
    wb.set_properties({"title": output.name, "author": "Stathis Devves"})
    fm = BookFormats(wb)

    readme = wb.add_worksheet("README")
    title(readme, fm, "Comparative summary: UC1 and UC2", "Consolidated comparison of six independently generated 8,760-hour workbooks.", 3)
    readme_rows = [
        ("Scope", "UC1 private EV and UC2 EV taxi; modes m=0, m=1 and m=2"),
        ("Hourly files", "Six separate workbooks preserve all 8,760 hourly rows"),
        ("Equations", "M-EQ01–M-EQ65; M-EQ63–M-EQ65 are the three production-externality additions"),
        ("Price year", "2025 wholesale prices from Input Data(8).xlsx"),
        ("IR", "A calculated mode may be rejected when its EV benefit is below the strongest outside option"),
        ("V2G", "Zero activation can be a valid economic result under the base calibration"),
        ("Runtime", "Fully Colab-compatible Python exporter"),
    ]
    for r, (key, value) in enumerate(readme_rows, 3):
        readme.write(r, 0, key, fm.label)
        readme.write(r, 1, value, fm.text_border)
    readme.set_column(0, 0, 24)
    readme.set_column(1, 1, 92)

    ws = wb.add_worksheet("Scenario_Comparison")
    title(ws, fm, "Comparative summary: UC1 and UC2", "Consolidated annual comparison of the six use-case/mode combinations.", 24)
    headers = ["Use_Case", "Mode", "Description", "Grid_Import_kWh", "Grid_Export_kWh", "EV_Charge_kWh", "EV_Discharge_kWh", "P2P_Buy_kWh", "P2P_Sell_kWh", "V2G_Reserve_kW-h", "V2G_Activation_kWh", "EV_Net_Benefit_EUR", "Prosumer_Net_Benefit_EUR", "DSO_Net_Benefit_EUR", "Real_System_Cost_EUR", "Social_Welfare_EUR", "CO2_kg", "Carbon_Cost_EUR", "Network_Cost_EUR", "Avoided_Externality_EUR", "EV_EFC", "SCR", "SSR", "P2P_VWAP_EUR_kWh", "IR_Flag"]
    write_headers(ws, fm, 3, headers)
    keys = ["grid_import", "grid_export", "ev_charge", "ev_discharge", "p2p_buy", "p2p_sell", "reserve", "activation", "ev_benefit", "prosumer_benefit", "dso_benefit", "real_cost", "welfare", "co2", "carbon", "network", "avoided", "efc", "scr", "ssr", "p2p_vwap", "ir"]
    for r, (case_id, mode, summary) in enumerate(all_rows, 4):
        ws.write_row(r, 0, [case_id, mode, mode_description(mode)])
        for c, key in enumerate(keys, 3):
            write_value(ws, r, c, summary[key], fm.percent if key in ("scr", "ssr") else fm.num4 if key == "p2p_vwap" else fm.num2)
    ws.set_column(0, 1, 12)
    ws.set_column(2, 2, 42)
    ws.set_column(3, 24, 19)
    ws.freeze_panes(4, 3)

    monthly = wb.add_worksheet("Monthly_Comparison")
    title(monthly, fm, "Monthly comparison index", "Monthly values are retained in each detailed workbook; this sheet maps every scenario to its source.", 4)
    write_headers(monthly, fm, 3, ["Use_Case", "Mode", "Workbook", "Source sheet", "Aggregation rule [M-EQ62]"])
    for r, (case_id, mode, _) in enumerate(all_rows, 4):
        label = "Private_EV" if case_id == "UC1" else "EV_Taxi"
        monthly.write_row(r, 0, [case_id, mode, f"{case_id}_m{mode}_{label}_8760_M_EQ01_65.xlsx", "Monthly_Summary", "SUMIFS by Month_ID; ratio-of-sums; volume-weighted price"])
    monthly.set_column(0, 1, 14)
    monthly.set_column(2, 4, 48)

    stakeholder = wb.add_worksheet("Stakeholder_Comparison")
    title(stakeholder, fm, "Stakeholder welfare decomposition", "Internal P2P/V2G transfers allocate benefits but do not independently create social welfare.", 6)
    write_headers(stakeholder, fm, 3, ["Use_Case", "Mode", "EV owner", "Prosumer", "DSO", "Social welfare", "IR"])
    for r, (case_id, mode, summary) in enumerate(all_rows, 4):
        stakeholder.write_row(r, 0, [case_id, mode])
        for c, key in enumerate(("ev_benefit", "prosumer_benefit", "dso_benefit", "welfare"), 2):
            stakeholder.write(r, c, summary[key], fm.num2)
        stakeholder.write(r, 6, summary["ir"])
    stakeholder.set_column(0, 1, 14)
    stakeholder.set_column(2, 6, 23)

    cooperative = wb.add_worksheet("Cooperative_Game")
    title(cooperative, fm, "Cooperative-game diagnostic comparison", "Values are calculated from coalition-specific rolling optimisations; convexity and core status are tested.", 7)
    write_headers(cooperative, fm, 3, ["Use_Case", "Grand_Coalition_v", "Shapley_P", "Shapley_E", "Shapley_D", "Core_Nonempty", "Convex", "Clan_Convex"])
    for r, case_id in enumerate(("UC1", "UC2"), 4):
        q = model["cases"][case_id]["coalition"]
        cooperative.write_row(r, 0, [case_id, q["values"]["DEP"], q["shapley"]["P"], q["shapley"]["E"], q["shapley"]["D"], int(q["al"]["core_nonempty"]), int(q["convexity"]["is_convex"]), int(q["clan"]["convexity"]["is_convex"])], fm.num2)
    cooperative.set_column(0, 7, 22)

    resilience = wb.add_worksheet("Resilience_Comparison")
    title(resilience, fm, "Resilience comparison", "Event-level M-EQ53–M-EQ55 results remain in each detailed workbook.", 4)
    write_headers(resilience, fm, 3, ["Use_Case", "Mode", "Events", "Workbook sheet", "Interpretation"])
    for r, (case_id, mode, _) in enumerate(all_rows, 4):
        resilience.write_row(r, 0, [case_id, mode, "Winter_24h; Summer_24h", "Resilience", "Sequential backup-energy depletion; VOLL and critical-load share editable"])
    resilience.set_column(0, 1, 14)
    resilience.set_column(2, 4, 50)

    validation = wb.add_worksheet("Validation")
    title(validation, fm, "Consolidated status", "Physical validation is performed in each detailed workbook; IR is reported separately.", 5)
    write_headers(validation, fm, 3, ["Use_Case", "Mode", "Physical checks", "Formula structure", "IR", "Detailed workbook"])
    for r, (case_id, mode, summary) in enumerate(all_rows, 4):
        label = "Private_EV" if case_id == "UC1" else "EV_Taxi"
        validation.write_row(r, 0, [case_id, mode, "PASS" if summary["max_balance"] <= 1e-6 else "FAIL", "M-EQ01–M-EQ65 mapped", summary["ir"], f"{case_id}_m{mode}_{label}_8760_M_EQ01_65.xlsx"])
    validation.set_column(0, 1, 14)
    validation.set_column(2, 5, 36)

    eq = wb.add_worksheet("Equation_Map")
    title(eq, fm, "Numbered methodology equation map", "Common equation definitions used in all six detailed workbooks.", 5)
    write_headers(eq, fm, 3, ["Equation ID", "Title", "Equation", "Purpose", "Excel / code implementation", "Literature basis"])
    for r, equation in enumerate(model["equations"], 4):
        for c, key in enumerate(("id", "title", "formula", "purpose", "implementation", "literature")):
            eq.write(r, c, equation[key], fm.text)
    eq.set_column(0, 0, 13)
    eq.set_column(1, 1, 36)
    eq.set_column(2, 5, 54)
    wb.close()
    print(f"Exported comparative summary: {output}", flush=True)
    return output


def main() -> None:
    if len(sys.argv) not in (4, 5):
        raise SystemExit("Usage: build_ev_workbooks_colab.py <model.json> <output-dir> <UC1|UC2|SUMMARY> [0|1|2]")
    model_path, output_dir, target = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3].upper()
    output_dir.mkdir(parents=True, exist_ok=True)
    model = json.loads(model_path.read_text(encoding="utf-8"))
    if target == "SUMMARY":
        build_comparative(model, output_dir)
        return
    if target not in ("UC1", "UC2") or len(sys.argv) != 5 or int(sys.argv[4]) not in (0, 1, 2):
        raise SystemExit("Detailed export requires target UC1 or UC2 and mode 0, 1 or 2")
    build_mode_workbook(model, output_dir, target, int(sys.argv[4]))


if __name__ == "__main__":
    main()


Writing /content/EV_P2P_V2G_Self_Contained_Code/build_ev_workbooks_colab.py


In [11]:
%%writefile /content/EV_P2P_V2G_Self_Contained_Code/run_colab_pipeline.py
from __future__ import annotations

import argparse
import json
import subprocess
import sys
from pathlib import Path
from zipfile import BadZipFile, ZipFile

from openpyxl import load_workbook

from extract_docx_spec import extract_text
from extract_ev_sources_colab import extract_workbook


DETAILED_SHEETS = {
    "README", "Assumptions", "Raw_Input_2025", "Hourly_Output", "Formula_Audit",
    "Monthly_Summary", "Annual_Summary", "Cooperative_Game", "Resilience",
    "Price_Transmission", "Validation", "Equation_Map",
}
SUMMARY_SHEETS = {
    "README", "Scenario_Comparison", "Monthly_Comparison", "Stakeholder_Comparison",
    "Cooperative_Game", "Resilience_Comparison", "Validation", "Equation_Map",
}


def run(command: list[str]) -> None:
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True)


def validate_xlsx(path: Path, summary: bool = False) -> None:
    try:
        with ZipFile(path) as archive:
            bad_member = archive.testzip()
            if bad_member:
                raise ValueError(f"Corrupt XLSX member: {bad_member}")
    except BadZipFile as exc:
        raise ValueError(f"Invalid XLSX archive: {path}") from exc

    workbook = load_workbook(path, read_only=True, data_only=False)
    required = SUMMARY_SHEETS if summary else DETAILED_SHEETS
    missing = required.difference(workbook.sheetnames)
    if missing:
        raise ValueError(f"Missing sheets in {path.name}: {sorted(missing)}")
    if not summary:
        if workbook["Hourly_Output"].max_row != 8764:
            raise ValueError(f"Expected 8,760 hourly rows in {path.name}")
        if workbook["Hourly_Output"].max_column != 70:
            raise ValueError(f"Expected 70 hourly columns in {path.name}")
    equation_map = workbook["Equation_Map"]
    ids = [equation_map.cell(row, 1).value for row in range(5, 70)]
    expected = [f"M-EQ{i:02d}" for i in range(1, 66)]
    if ids != expected:
        raise ValueError(f"Incomplete M-EQ01–M-EQ65 map in {path.name}")
    workbook.close()
    print(f"Validated: {path.name}", flush=True)


def reusable_workbook(path: Path, summary: bool = False) -> bool:
    if not path.exists():
        return False
    try:
        validate_xlsx(path, summary=summary)
        print(f"RESUME: keeping completed workbook {path.name}", flush=True)
        return True
    except Exception as exc:
        print(f"REBUILD: existing workbook is incomplete or invalid ({path.name}): {exc}", flush=True)
        return False


def main() -> None:
    parser = argparse.ArgumentParser(description="Run the complete Colab-compatible EV–P2P–V2G pipeline.")
    parser.add_argument("--input", required=True, type=Path, help="Input Data(8).xlsx")
    parser.add_argument("--spec", required=True, type=Path, help="M-EQ specification DOCX")
    parser.add_argument("--output", required=True, type=Path, help="Output directory")
    parser.add_argument("--resume", action="store_true", help="Reuse valid simulation checkpoints and completed workbooks")
    parser.add_argument("--progress-every", type=int, default=500, help="Report optimisation progress every N rolling hours")
    args = parser.parse_args()

    script_dir = Path(__file__).resolve().parent
    output_dir = args.output.resolve()
    intermediate = output_dir / "_intermediate"
    output_dir.mkdir(parents=True, exist_ok=True)
    intermediate.mkdir(parents=True, exist_ok=True)

    input_json = intermediate / "input_values.json"
    spec_text = intermediate / "equation_spec.txt"
    model_json = intermediate / "ev_model.json"

    input_json.write_text(json.dumps(extract_workbook(args.input.resolve()), ensure_ascii=False), encoding="utf-8")
    spec_text.write_text(extract_text(args.spec.resolve()), encoding="utf-8")
    print("Inputs and M-EQ specification extracted.", flush=True)

    model_command = [
        sys.executable, str(script_dir / "prepare_ev_model.py"),
        str(input_json), str(spec_text), str(model_json),
        "--checkpoint-dir", str(intermediate / "checkpoints"),
        "--progress-every", str(args.progress_every),
    ]
    if args.resume:
        model_command.append("--resume")
    run(model_command)

    for case_id in ("UC1", "UC2"):
        label = "Private_EV" if case_id == "UC1" else "EV_Taxi"
        for mode in (0, 1, 2):
            workbook_path = output_dir / f"{case_id}_m{mode}_{label}_8760_M_EQ01_65.xlsx"
            if args.resume and reusable_workbook(workbook_path):
                continue
            run([sys.executable, str(script_dir / "build_ev_workbooks_colab.py"), str(model_json), str(output_dir), case_id, str(mode)])
            validate_xlsx(workbook_path)

    summary_path = output_dir / "Comparative_Summary_UC1_UC2_M_EQ01_65.xlsx"
    if not (args.resume and reusable_workbook(summary_path, summary=True)):
        run([sys.executable, str(script_dir / "build_ev_workbooks_colab.py"), str(model_json), str(output_dir), "SUMMARY"])
        validate_xlsx(summary_path, summary=True)
    print(f"Complete. Results: {output_dir}", flush=True)


if __name__ == "__main__":
    main()


Writing /content/EV_P2P_V2G_Self_Contained_Code/run_colab_pipeline.py


## 5. Verify the embedded code

This fast integrity check confirms that every embedded module was written exactly as packaged in the notebook before the long optimization starts.


In [12]:
import hashlib

expected_hashes = {'extract_docx_spec.py': '02ff3580c785bbf82744dc8ffe0f8c1a603cac991f1d1a55ed9bdb0101cde350', 'extract_ev_sources_colab.py': '9a9bc4c5186a496c1d7f1d54d688c9afce569cc5589394db92c125db1331fec1', 'prepare_ev_model.py': '613f7fc4b52fa56d7df818e8b5bc4aa6208d0db7db39b882ee9af38f9eb81cf9', 'build_ev_workbooks_colab.py': '3acfa5d3ed984ec27787b5b46023193aca3a2b0ead303cc5e99516ac89482ffa', 'run_colab_pipeline.py': '5b811387b2af2fb363e3cd25b22f1b6d3bd4117df100210e3e5e573fc051eca7'}
for filename, expected in expected_hashes.items():
    path = code_dir / filename
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Embedded-code integrity check failed: {filename}')
    print('Verified:', filename)


Verified: extract_docx_spec.py
Verified: extract_ev_sources_colab.py
Verified: prepare_ev_model.py
Verified: build_ev_workbooks_colab.py
Verified: run_colab_pipeline.py


## 6. Run or resume

Run this cell to start the complete pipeline. Progress is reported every 500 rolling hours. A checkpoint is saved after each completed case/mode calculation. If interrupted, rerun Sections 1–6; valid checkpoints and completed workbooks are reused automatically.


In [13]:
command = [
    sys.executable, str(code_dir / 'run_colab_pipeline.py'),
    '--input', str(input_xlsx),
    '--spec', str(spec_docx),
    '--output', str(results_dir),
    '--resume',
    '--progress-every', '500',
]
print('Starting or resuming the EV–P2P–V2G pipeline...')
subprocess.run(command, check=True)
print('All workbooks generated and validated.')


Starting or resuming the EV–P2P–V2G pipeline...
All workbooks generated and validated.


## 7. Optional checkpoint status

Run this cell after an interruption, or whenever the optimization cell is not running, to see which scenario checkpoints have been saved.


In [14]:
import json

manifest = results_dir / '_intermediate' / 'checkpoints' / 'checkpoint_manifest.json'
if manifest.exists():
    status = json.loads(manifest.read_text(encoding='utf-8'))
    print(f"Completed checkpoints: {status['completed_count']}/{status['total_runs']}")
    for item in status['completed']:
        print('  ✓', item)
else:
    print('No completed checkpoint has been saved yet.')


Completed checkpoints: 12/12
  ✓ UC1_m0_der.json
  ✓ UC1_m1_der.json
  ✓ UC1_m2_der.json
  ✓ UC1_m0_no_der.json
  ✓ UC1_m1_no_der.json
  ✓ UC1_m2_no_der.json
  ✓ UC2_m0_der.json
  ✓ UC2_m1_der.json
  ✓ UC2_m2_der.json
  ✓ UC2_m0_no_der.json
  ✓ UC2_m1_no_der.json
  ✓ UC2_m2_no_der.json


## 8. Download the final Excel workbooks

Run this cell only after Section 6 prints `All workbooks generated and validated.` It creates a ZIP containing only the seven final `.xlsx` files; persistent checkpoints remain in Drive.


In [15]:
from zipfile import ZIP_DEFLATED, ZipFile

workbooks = sorted(results_dir.glob('*.xlsx'))
if len(workbooks) != 7:
    raise RuntimeError(f'Expected 7 final workbooks, but found {len(workbooks)}. Complete Section 6 first.')

archive_path = Path('/content/EV_P2P_V2G_Results.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for workbook in workbooks:
        archive.write(workbook, workbook.name)

print('Prepared:', archive_path.name)
files.download(str(archive_path))


Prepared: EV_P2P_V2G_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>